In [1]:
# ==================================================================================================
# CISLR — L18A
# EXISTING-DATA DISCOVERY + VIDEO INTEGRITY AUDIT
# FOR THE FROZEN L17 10-SIGN DEPLOYMENT VOCABULARY
# ==================================================================================================
#
# PURPOSE
# -------
# L18A DOES NOT TRAIN A MODEL.
# L18A DOES NOT CHANGE L16 OR L17.
# L18A DOES NOT RUN TEST INFERENCE.
#
# It:
#
#   1. Loads the exact frozen L17 10-sign vocabulary.
#   2. Scans dataset.csv for EVERY occurrence of those glosses.
#   3. Resolves every UID to its real CISLR MP4.
#   4. Reads only video metadata for integrity:
#        - readable?
#        - frame count
#        - FPS
#        - duration
#        - resolution
#   5. Cross-references each UID with the frozen L9 split.
#   6. Labels every video as:
#
#          ORIGINAL_L9_TRAIN
#          ORIGINAL_L9_VALIDATION
#          ORIGINAL_L9_TEST
#          OUTSIDE_L9_CONTROLLED_SPLIT
#
#   7. Creates safe candidate manifests.
#   8. Does NOT silently put L9 validation/test into training.
#   9. Saves hashes and a full audit report.
#
#
# IMPORTANT
# ---------
# "OUTSIDE_L9_CONTROLLED_SPLIT" does NOT automatically mean training-safe.
#
# L18B will decide how the new 10-class experiment is split.
#
# ==================================================================================================


# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

import os
import cv2
import json
import math
import random
import hashlib
import platform

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# ==================================================================================================
# 2. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)


# --------------------------------------------------------------------------------------------------
# ORIGINAL CISLR
# --------------------------------------------------------------------------------------------------

DATASET_CSV = Path(
    "/home/dipshikha/Music/cao/dataset.csv"
)

VIDEO_DIR = Path(
    "/home/dipshikha/Music/cao/CISLR_v1.5-a_videos"
)


# --------------------------------------------------------------------------------------------------
# L9
# --------------------------------------------------------------------------------------------------

L9_ROOT = (
    ROOT /
    "audit/l9_split_freeze"
)

L9_TRAIN = (
    L9_ROOT /
    "splits/l9_train_manifest.csv"
)

L9_VAL = (
    L9_ROOT /
    "splits/l9_validation_manifest.csv"
)

L9_TEST = (
    L9_ROOT /
    "splits/l9_test_manifest.csv"
)


# --------------------------------------------------------------------------------------------------
# L17
# --------------------------------------------------------------------------------------------------

L17_ROOT = (
    ROOT /
    "audit/l17_deployment_vocabulary"
)

L17_VOCAB_JSON = (
    L17_ROOT /
    "vocabulary/l17_10_sign_vocabulary.json"
)

L17_MASTER_FREEZE = (
    L17_ROOT /
    "L17_MASTER_FREEZE.json"
)


# --------------------------------------------------------------------------------------------------
# L18A OUTPUT
# --------------------------------------------------------------------------------------------------

L18A_ROOT = (
    ROOT /
    "audit/l18a_existing_cislr_discovery"
)

MANIFEST_DIR = (
    L18A_ROOT /
    "manifests"
)

REPORT_DIR = (
    L18A_ROOT /
    "reports"
)

HASH_DIR = (
    L18A_ROOT /
    "hashes"
)


for path in [
    L18A_ROOT,
    MANIFEST_DIR,
    REPORT_DIR,
    HASH_DIR,
]:

    path.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 3. OUTPUT PATHS
# ==================================================================================================

ALL_SELECTED_MANIFEST = (
    MANIFEST_DIR /
    "l18a_all_10sign_cislr_videos.csv"
)

TRAIN_MANIFEST = (
    MANIFEST_DIR /
    "l18a_original_l9_train.csv"
)

VAL_MANIFEST = (
    MANIFEST_DIR /
    "l18a_original_l9_validation.csv"
)

TEST_MANIFEST = (
    MANIFEST_DIR /
    "l18a_original_l9_test.csv"
)

OUTSIDE_MANIFEST = (
    MANIFEST_DIR /
    "l18a_outside_l9_controlled_split.csv"
)

USABLE_MANIFEST = (
    MANIFEST_DIR /
    "l18a_all_readable_videos.csv"
)

SUMMARY_CSV = (
    REPORT_DIR /
    "l18a_gloss_summary.csv"
)

SUMMARY_JSON = (
    REPORT_DIR /
    "l18a_summary.json"
)

HASH_MANIFEST = (
    HASH_DIR /
    "l18a_sha256_manifest.json"
)

MASTER_FREEZE = (
    L18A_ROOT /
    "L18A_MASTER_FREEZE.json"
)


# ==================================================================================================
# 4. HELPERS
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def save_json(
    obj,
    path
):

    with open(
        path,
        "w"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )


def first_existing_column(
    df,
    candidates
):

    for column in candidates:

        if column in df.columns:

            return column

    return None


def clean_string(
    value
):

    if pd.isna(value):

        return ""

    return str(
        value
    ).strip()


def normalize_uid(
    value
):

    uid = clean_string(
        value
    )

    if uid.lower().endswith(
        ".mp4"
    ):

        uid = uid[:-4]

    return uid


def normalize_gloss(
    value
):

    return clean_string(
        value
    ).lower()


def resolve_video_path(
    uid
):

    # ----------------------------------------------------------------------------------------------
    # Primary expected path
    # ----------------------------------------------------------------------------------------------

    direct = (
        VIDEO_DIR /
        f"{uid}.mp4"
    )

    if direct.exists():

        return direct


    # ----------------------------------------------------------------------------------------------
    # Sometimes metadata may already contain extension
    # ----------------------------------------------------------------------------------------------

    direct_raw = (
        VIDEO_DIR /
        uid
    )

    if direct_raw.exists():

        return direct_raw


    return None


def inspect_video(
    video_path
):

    result = {

        "video_readable":
            False,

        "frame_count":
            0,

        "fps":
            0.0,

        "duration_seconds":
            0.0,

        "width":
            0,

        "height":
            0,

        "file_size_mb":
            0.0,

        "first_frame_readable":
            False,
    }


    if (
        video_path is None
        or
        not Path(
            video_path
        ).exists()
    ):

        return result


    video_path = Path(
        video_path
    )


    result[
        "file_size_mb"
    ] = (

        video_path.stat().st_size
        /
        (
            1024 * 1024
        )
    )


    cap = cv2.VideoCapture(
        str(
            video_path
        )
    )


    if not cap.isOpened():

        cap.release()

        return result


    frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )

    width = int(
        cap.get(
            cv2.CAP_PROP_FRAME_WIDTH
        )
    )

    height = int(
        cap.get(
            cv2.CAP_PROP_FRAME_HEIGHT
        )
    )


    ok, _ = cap.read()

    cap.release()


    duration = 0.0

    if (
        np.isfinite(fps)
        and
        fps > 0
        and
        frame_count > 0
    ):

        duration = (
            frame_count /
            fps
        )


    result.update(
        {

            "video_readable":
                bool(
                    ok
                    and
                    frame_count > 0
                ),

            "frame_count":
                int(
                    frame_count
                ),

            "fps":
                float(
                    fps
                    if np.isfinite(fps)
                    else 0.0
                ),

            "duration_seconds":
                float(
                    duration
                ),

            "width":
                int(
                    width
                ),

            "height":
                int(
                    height
                ),

            "first_frame_readable":
                bool(
                    ok
                ),
        }
    )


    return result


# ==================================================================================================
# 5. HEADER
# ==================================================================================================

print("=" * 140)
print("CISLR — L18A")
print("EXISTING CISLR 10-SIGN DATA DISCOVERY + VIDEO AUDIT")
print("=" * 140)

print()

print(
    "Python:",
    platform.python_version()
)

print(
    "Seed:",
    SEED
)

print(
    "Dataset CSV:"
)

print(
    DATASET_CSV
)

print(
    "Video directory:"
)

print(
    VIDEO_DIR
)


# ==================================================================================================
# 6. REQUIRED ARTIFACT CHECK
# ==================================================================================================

print("\n" + "=" * 140)
print("1. REQUIRED ARTIFACT CHECK")
print("=" * 140)


required = {

    "dataset.csv":
        DATASET_CSV,

    "CISLR video directory":
        VIDEO_DIR,

    "L9 train":
        L9_TRAIN,

    "L9 validation":
        L9_VAL,

    "L9 test":
        L9_TEST,

    "L17 vocabulary":
        L17_VOCAB_JSON,

    "L17 master freeze":
        L17_MASTER_FREEZE,
}


for name, path in (
    required.items()
):

    exists = path.exists()

    print(
        f"{name:25s}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    if not exists:

        raise FileNotFoundError(
            path
        )


print()

print(
    "[PASS] All required artifacts exist."
)


# ==================================================================================================
# 7. LOAD FROZEN L17 VOCABULARY
# ==================================================================================================

print("\n" + "=" * 140)
print("2. LOADING FROZEN L17 VOCABULARY")
print("=" * 140)


with open(
    L17_VOCAB_JSON,
    "r"
) as f:

    vocab_data = json.load(
        f
    )


if int(
    vocab_data[
        "num_classes"
    ]
) != 10:

    raise RuntimeError(
        "Expected exactly 10 L17 classes."
    )


signs = []


for item in (
    vocab_data[
        "signs"
    ]
):

    signs.append(
        {

            "deployment_class_id":
                int(
                    item[
                        "deployment_class_id"
                    ]
                ),

            "gloss":
                normalize_gloss(
                    item[
                        "gloss"
                    ]
                ),
        }
    )


signs = sorted(
    signs,
    key=lambda x:
        x[
            "deployment_class_id"
        ]
)


if len(
    signs
) != 10:

    raise RuntimeError(
        "L17 vocabulary does not contain exactly 10 signs."
    )


selected_glosses = [

    item[
        "gloss"
    ]

    for item in signs
]


gloss_to_deployment_id = {

    item[
        "gloss"
    ]:
        item[
            "deployment_class_id"
        ]

    for item in signs
}


for item in signs:

    print(
        f"{item['deployment_class_id']:2d}  "
        f"{item['gloss']}"
    )


print()

print(
    "[PASS] L17 vocabulary loaded."
)


# ==================================================================================================
# 8. LOAD dataset.csv
# ==================================================================================================

print("\n" + "=" * 140)
print("3. LOADING ORIGINAL CISLR METADATA")
print("=" * 140)


dataset_df = pd.read_csv(
    DATASET_CSV
)


print(
    "Rows:",
    len(
        dataset_df
    )
)

print(
    "Columns:",
    list(
        dataset_df.columns
    )
)


uid_col = first_existing_column(
    dataset_df,
    [
        "uid",
        "uid_clean",
        "video_id",
        "id",
    ]
)


gloss_col = first_existing_column(
    dataset_df,
    [
        "gloss",
        "authoritative_gloss",
        "feature_gloss",
        "label",
    ]
)


if uid_col is None:

    raise RuntimeError(
        "Could not determine UID column in dataset.csv."
    )


if gloss_col is None:

    raise RuntimeError(
        "Could not determine gloss column in dataset.csv."
    )


print()

print(
    "UID column:",
    uid_col
)

print(
    "Gloss column:",
    gloss_col
)


dataset_df[
    "_uid"
] = (
    dataset_df[
        uid_col
    ]
    .apply(
        normalize_uid
    )
)


dataset_df[
    "_gloss"
] = (
    dataset_df[
        gloss_col
    ]
    .apply(
        normalize_gloss
    )
)


# ==================================================================================================
# 9. SELECT ALL OCCURRENCES OF THE 10 GLOSSES
# ==================================================================================================

print("\n" + "=" * 140)
print("4. DISCOVERING ALL 10-SIGN CISLR VIDEOS")
print("=" * 140)


selected_df = (

    dataset_df[
        dataset_df[
            "_gloss"
        ].isin(
            selected_glosses
        )
    ]
    .copy()
)


selected_df[
    "deployment_class_id"
] = (

    selected_df[
        "_gloss"
    ]
    .map(
        gloss_to_deployment_id
    )
    .astype(int)
)


selected_df[
    "gloss"
] = (
    selected_df[
        "_gloss"
    ]
)


selected_df[
    "uid"
] = (
    selected_df[
        "_uid"
    ]
)


print(
    "Matching metadata rows:",
    len(
        selected_df
    )
)


print()

print(
    "Raw support by gloss:"
)


raw_counts = (

    selected_df
    .groupby(
        [
            "deployment_class_id",
            "gloss"
        ]
    )
    .size()
    .rename(
        "metadata_count"
    )
    .reset_index()
    .sort_values(
        "deployment_class_id"
    )
)


print(
    raw_counts.to_string(
        index=False
    )
)


# ==================================================================================================
# 10. DUPLICATE UID AUDIT
# ==================================================================================================

print("\n" + "=" * 140)
print("5. UID DUPLICATE AUDIT")
print("=" * 140)


duplicate_uid_mask = (
    selected_df[
        "uid"
    ].duplicated(
        keep=False
    )
)


duplicate_rows = (
    selected_df[
        duplicate_uid_mask
    ]
    .copy()
)


print(
    "Duplicate UID rows:",
    len(
        duplicate_rows
    )
)


if len(
    duplicate_rows
) > 0:

    print()

    print(
        duplicate_rows[
            [
                "uid",
                "gloss",
                "deployment_class_id"
            ]
        ]
        .sort_values(
            "uid"
        )
        .to_string(
            index=False
        )
    )


# ==================================================================================================
# 11. LOAD L9 SPLITS
# ==================================================================================================

print("\n" + "=" * 140)
print("6. LOADING FROZEN L9 SPLITS")
print("=" * 140)


l9_train = pd.read_csv(
    L9_TRAIN
)

l9_val = pd.read_csv(
    L9_VAL
)

l9_test = pd.read_csv(
    L9_TEST
)


print(
    "L9 train:",
    len(
        l9_train
    )
)

print(
    "L9 validation:",
    len(
        l9_val
    )
)

print(
    "L9 test:",
    len(
        l9_test
    )
)


def get_uid_set(
    df
):

    local_uid_col = (
        first_existing_column(
            df,
            [
                "uid",
                "uid_clean",
                "video_id",
                "id"
            ]
        )
    )


    if local_uid_col is None:

        raise RuntimeError(
            "Could not identify UID column in L9 manifest."
        )


    return set(

        df[
            local_uid_col
        ]
        .apply(
            normalize_uid
        )
        .tolist()
    )


train_uids = get_uid_set(
    l9_train
)

val_uids = get_uid_set(
    l9_val
)

test_uids = get_uid_set(
    l9_test
)


# ==================================================================================================
# 12. VERIFY ORIGINAL SPLIT DISJOINTNESS
# ==================================================================================================

train_val_overlap = (
    train_uids
    &
    val_uids
)

train_test_overlap = (
    train_uids
    &
    test_uids
)

val_test_overlap = (
    val_uids
    &
    test_uids
)


print()

print(
    "Train ∩ Validation:",
    len(
        train_val_overlap
    )
)

print(
    "Train ∩ Test:",
    len(
        train_test_overlap
    )
)

print(
    "Validation ∩ Test:",
    len(
        val_test_overlap
    )
)


if (
    train_val_overlap
    or
    train_test_overlap
    or
    val_test_overlap
):

    raise RuntimeError(
        "L9 split overlap detected."
    )


print()

print(
    "[PASS] Original L9 partitions remain disjoint."
)


# ==================================================================================================
# 13. ASSIGN ORIGINAL L9 PROVENANCE
# ==================================================================================================

def determine_origin(
    uid
):

    if uid in train_uids:

        return "ORIGINAL_L9_TRAIN"

    if uid in val_uids:

        return "ORIGINAL_L9_VALIDATION"

    if uid in test_uids:

        return "ORIGINAL_L9_TEST"

    return "OUTSIDE_L9_CONTROLLED_SPLIT"


selected_df[
    "l9_origin"
] = (
    selected_df[
        "uid"
    ]
    .apply(
        determine_origin
    )
)


print()

print(
    selected_df[
        "l9_origin"
    ]
    .value_counts()
    .to_string()
)


# ==================================================================================================
# 14. RESOLVE VIDEO FILES
# ==================================================================================================

print("\n" + "=" * 140)
print("7. RESOLVING VIDEO FILES")
print("=" * 140)


video_paths = []

video_exists = []


for uid in (
    selected_df[
        "uid"
    ].tolist()
):

    path = resolve_video_path(
        uid
    )


    if path is None:

        video_paths.append(
            ""
        )

        video_exists.append(
            False
        )

    else:

        video_paths.append(
            str(
                path
            )
        )

        video_exists.append(
            True
        )


selected_df[
    "video_path"
] = video_paths


selected_df[
    "video_exists"
] = video_exists


print(
    "Resolved:",
    int(
        selected_df[
            "video_exists"
        ].sum()
    ),
    "/",
    len(
        selected_df
    )
)


print(
    "Missing:",
    int(
        (
            ~selected_df[
                "video_exists"
            ]
        ).sum()
    )
)


# ==================================================================================================
# 15. VIDEO INTEGRITY AUDIT
#
# This reads only enough to verify the video can be opened.
#
# It does NOT run recognition.
# ==================================================================================================

print("\n" + "=" * 140)
print("8. VIDEO INTEGRITY AUDIT")
print("=" * 140)


audit_results = []


total = len(
    selected_df
)


for position, (
    idx,
    row
) in enumerate(
    selected_df.iterrows(),
    start=1
):

    path_text = (
        row[
            "video_path"
        ]
    )


    if path_text:

        info = inspect_video(
            Path(
                path_text
            )
        )

    else:

        info = {

            "video_readable":
                False,

            "frame_count":
                0,

            "fps":
                0.0,

            "duration_seconds":
                0.0,

            "width":
                0,

            "height":
                0,

            "file_size_mb":
                0.0,

            "first_frame_readable":
                False,
        }


    audit_results.append(
        info
    )


    print(
        f"\r"
        f"Auditing video "
        f"{position:3d}/{total:3d} "
        f"| {row['gloss']:<15s} "
        f"| {row['uid']:<15s}",
        end="",
        flush=True
    )


print()


audit_df = pd.DataFrame(
    audit_results,
    index=selected_df.index
)


for column in (
    audit_df.columns
):

    selected_df[
        column
    ] = (
        audit_df[
            column
        ]
    )


# ==================================================================================================
# 16. BASIC USABILITY
# ==================================================================================================

selected_df[
    "basic_usable"
] = (

    selected_df[
        "video_exists"
    ]

    &

    selected_df[
        "video_readable"
    ]

    &

    (
        selected_df[
            "frame_count"
        ]
        >
        0
    )

    &

    (
        selected_df[
            "width"
        ]
        >
        0
    )

    &

    (
        selected_df[
            "height"
        ]
        >
        0
    )
)


print()

print(
    "Readable:",
    int(
        selected_df[
            "video_readable"
        ].sum()
    )
)

print(
    "Basic usable:",
    int(
        selected_df[
            "basic_usable"
        ].sum()
    )
)


# ==================================================================================================
# 17. SAFETY / USAGE LABEL
#
# This deliberately does NOT automatically declare outside-L9 videos
# to be training data.
# ==================================================================================================

def usage_label(
    row
):

    if not bool(
        row[
            "basic_usable"
        ]
    ):

        return "REJECT_UNREADABLE"


    origin = row[
        "l9_origin"
    ]


    if origin == "ORIGINAL_L9_TRAIN":

        return "TRAIN_ELIGIBLE_EXISTING"


    if origin == "ORIGINAL_L9_VALIDATION":

        return "RESERVED_ORIGINAL_VALIDATION"


    if origin == "ORIGINAL_L9_TEST":

        return "LOCKED_ORIGINAL_TEST"


    return "UNASSIGNED_EXISTING_CISLR"


selected_df[
    "l18a_usage_status"
] = (
    selected_df.apply(
        usage_label,
        axis=1
    )
)


# ==================================================================================================
# 18. CLEAN OUTPUT TABLE
# ==================================================================================================

preferred_columns = [

    "deployment_class_id",
    "gloss",
    "uid",

    "l9_origin",
    "l18a_usage_status",

    "video_path",
    "video_exists",
    "video_readable",
    "basic_usable",

    "frame_count",
    "fps",
    "duration_seconds",
    "width",
    "height",
    "file_size_mb",
]


# Add original metadata fields afterward.

remaining_columns = [

    col

    for col in selected_df.columns

    if (
        col not in preferred_columns
        and
        col not in {
            "_uid",
            "_gloss"
        }
    )
]


selected_df = selected_df[
    preferred_columns
    +
    remaining_columns
]


selected_df = (

    selected_df
    .sort_values(
        [
            "deployment_class_id",
            "l9_origin",
            "uid"
        ]
    )
    .reset_index(
        drop=True
    )
)


# ==================================================================================================
# 19. SAVE MASTER MANIFEST
# ==================================================================================================

selected_df.to_csv(
    ALL_SELECTED_MANIFEST,
    index=False
)


# ==================================================================================================
# 20. SAVE PROVENANCE-SPECIFIC MANIFESTS
# ==================================================================================================

original_train_df = (

    selected_df[
        selected_df[
            "l9_origin"
        ]
        ==
        "ORIGINAL_L9_TRAIN"
    ]
    .copy()
)


original_val_df = (

    selected_df[
        selected_df[
            "l9_origin"
        ]
        ==
        "ORIGINAL_L9_VALIDATION"
    ]
    .copy()
)


original_test_df = (

    selected_df[
        selected_df[
            "l9_origin"
        ]
        ==
        "ORIGINAL_L9_TEST"
    ]
    .copy()
)


outside_df = (

    selected_df[
        selected_df[
            "l9_origin"
        ]
        ==
        "OUTSIDE_L9_CONTROLLED_SPLIT"
    ]
    .copy()
)


usable_df = (

    selected_df[
        selected_df[
            "basic_usable"
        ]
    ]
    .copy()
)


original_train_df.to_csv(
    TRAIN_MANIFEST,
    index=False
)

original_val_df.to_csv(
    VAL_MANIFEST,
    index=False
)

original_test_df.to_csv(
    TEST_MANIFEST,
    index=False
)

outside_df.to_csv(
    OUTSIDE_MANIFEST,
    index=False
)

usable_df.to_csv(
    USABLE_MANIFEST,
    index=False
)


# ==================================================================================================
# 21. PER-GLOSS SUMMARY
# ==================================================================================================

print("\n" + "=" * 140)
print("9. PER-GLOSS DATA SUMMARY")
print("=" * 140)


summary_rows = []


for item in signs:

    class_id = (
        item[
            "deployment_class_id"
        ]
    )

    gloss = (
        item[
            "gloss"
        ]
    )


    g = selected_df[
        selected_df[
            "deployment_class_id"
        ]
        ==
        class_id
    ]


    summary_rows.append(
        {

            "deployment_class_id":
                class_id,

            "gloss":
                gloss,

            "total_metadata":
                len(
                    g
                ),

            "video_exists":
                int(
                    g[
                        "video_exists"
                    ].sum()
                ),

            "readable":
                int(
                    g[
                        "video_readable"
                    ].sum()
                ),

            "usable":
                int(
                    g[
                        "basic_usable"
                    ].sum()
                ),

            "l9_train":
                int(
                    (
                        g[
                            "l9_origin"
                        ]
                        ==
                        "ORIGINAL_L9_TRAIN"
                    ).sum()
                ),

            "l9_validation":
                int(
                    (
                        g[
                            "l9_origin"
                        ]
                        ==
                        "ORIGINAL_L9_VALIDATION"
                    ).sum()
                ),

            "l9_test":
                int(
                    (
                        g[
                            "l9_origin"
                        ]
                        ==
                        "ORIGINAL_L9_TEST"
                    ).sum()
                ),

            "outside_l9":
                int(
                    (
                        g[
                            "l9_origin"
                        ]
                        ==
                        "OUTSIDE_L9_CONTROLLED_SPLIT"
                    ).sum()
                ),

            "mean_duration_seconds":
                float(
                    g.loc[
                        g[
                            "basic_usable"
                        ],
                        "duration_seconds"
                    ].mean()
                )
                if g[
                    "basic_usable"
                ].any()
                else 0.0,

            "mean_fps":
                float(
                    g.loc[
                        g[
                            "basic_usable"
                        ],
                        "fps"
                    ].mean()
                )
                if g[
                    "basic_usable"
                ].any()
                else 0.0,
        }
    )


gloss_summary = pd.DataFrame(
    summary_rows
)


gloss_summary.to_csv(
    SUMMARY_CSV,
    index=False
)


print()

print(
    gloss_summary.to_string(
        index=False
    )
)


# ==================================================================================================
# 22. GLOBAL COUNTS
# ==================================================================================================

print("\n" + "=" * 140)
print("10. GLOBAL DATA COUNTS")
print("=" * 140)


total_metadata = len(
    selected_df
)

total_existing = int(
    selected_df[
        "video_exists"
    ].sum()
)

total_readable = int(
    selected_df[
        "video_readable"
    ].sum()
)

total_usable = int(
    selected_df[
        "basic_usable"
    ].sum()
)


num_train = len(
    original_train_df
)

num_val = len(
    original_val_df
)

num_test = len(
    original_test_df
)

num_outside = len(
    outside_df
)


print(
    "10-sign metadata rows       :",
    total_metadata
)

print(
    "Videos found                :",
    total_existing
)

print(
    "Videos readable             :",
    total_readable
)

print(
    "Videos basic usable         :",
    total_usable
)

print()

print(
    "Original L9 TRAIN           :",
    num_train
)

print(
    "Original L9 VALIDATION      :",
    num_val
)

print(
    "Original L9 TEST            :",
    num_test
)

print(
    "Outside controlled L9       :",
    num_outside
)


# ==================================================================================================
# 23. IMPORTANT LEAKAGE CHECK
# ==================================================================================================

print("\n" + "=" * 140)
print("11. LEAKAGE SAFETY CHECK")
print("=" * 140)


train_uid_set = set(
    original_train_df[
        "uid"
    ]
)


val_uid_set = set(
    original_val_df[
        "uid"
    ]
)


test_uid_set = set(
    original_test_df[
        "uid"
    ]
)


assert len(
    train_uid_set
    &
    val_uid_set
) == 0


assert len(
    train_uid_set
    &
    test_uid_set
) == 0


assert len(
    val_uid_set
    &
    test_uid_set
) == 0


print(
    "[PASS] Selected L9 train/validation/test UIDs remain disjoint."
)

print(
    "[PASS] Original L9 validation has NOT been reassigned to training."
)

print(
    "[PASS] Original L9 test has NOT been reassigned to training."
)

print(
    "[PASS] Outside-L9 videos remain UNASSIGNED."
)


# ==================================================================================================
# 24. SAVE SUMMARY JSON
# ==================================================================================================

summary = {

    "stage":
        "L18A",

    "status":
        "COMPLETE",

    "purpose":
        (
            "Discover and audit every existing CISLR video "
            "belonging to the frozen L17 10-sign vocabulary."
        ),

    "seed":
        SEED,

    "num_classes":
        10,

    "frozen_vocabulary": {

        str(
            item[
                "deployment_class_id"
            ]
        ):
            item[
                "gloss"
            ]

        for item in signs
    },

    "counts": {

        "metadata_rows":
            total_metadata,

        "video_files_found":
            total_existing,

        "readable_videos":
            total_readable,

        "basic_usable_videos":
            total_usable,

        "original_l9_train":
            num_train,

        "original_l9_validation":
            num_val,

        "original_l9_test":
            num_test,

        "outside_l9_controlled_split":
            num_outside,
    },

    "scientific_rules": {

        "l9_validation_added_to_training":
            False,

        "l9_test_added_to_training":
            False,

        "outside_l9_automatically_assigned_to_training":
            False,

        "model_training_performed":
            False,

        "model_inference_performed":
            False,

        "l16_modified":
            False,

        "l17_modified":
            False,
    },

    "next_stage":
        (
            "L18B — define the restricted-vocabulary "
            "training/validation/deployment evaluation protocol"
        ),

    "timestamp":
        datetime.now().isoformat(),
}


save_json(
    summary,
    SUMMARY_JSON
)


# ==================================================================================================
# 25. HASH OUTPUT ARTIFACTS
# ==================================================================================================

print("\n" + "=" * 140)
print("12. ARTIFACT HASHING")
print("=" * 140)


artifact_paths = {

    "all_10sign_videos":
        ALL_SELECTED_MANIFEST,

    "original_l9_train":
        TRAIN_MANIFEST,

    "original_l9_validation":
        VAL_MANIFEST,

    "original_l9_test":
        TEST_MANIFEST,

    "outside_l9":
        OUTSIDE_MANIFEST,

    "all_readable":
        USABLE_MANIFEST,

    "gloss_summary":
        SUMMARY_CSV,

    "summary_json":
        SUMMARY_JSON,
}


hashes = {}


for name, path in (
    artifact_paths.items()
):

    digest = sha256_file(
        path
    )


    hashes[
        name
    ] = {

        "path":
            str(
                path
            ),

        "sha256":
            digest,
    }


    print(
        f"{name:28s}: "
        f"{digest}"
    )


save_json(
    hashes,
    HASH_MANIFEST
)


# ==================================================================================================
# 26. MASTER FREEZE
# ==================================================================================================

master_freeze = {

    "stage":
        "L18A",

    "status":
        "COMPLETE_AND_FROZEN",

    "num_classes":
        10,

    "vocabulary_source":
        str(
            L17_VOCAB_JSON
        ),

    "dataset_source":
        str(
            DATASET_CSV
        ),

    "video_source":
        str(
            VIDEO_DIR
        ),

    "counts": {

        "metadata":
            total_metadata,

        "found":
            total_existing,

        "readable":
            total_readable,

        "usable":
            total_usable,

        "l9_train":
            num_train,

        "l9_validation":
            num_val,

        "l9_test":
            num_test,

        "outside_l9":
            num_outside,
    },

    "l9_validation_reassigned":
        False,

    "l9_test_reassigned":
        False,

    "outside_l9_assignment":
        "UNASSIGNED",

    "training_performed":
        False,

    "inference_performed":
        False,

    "created_at":
        datetime.now().isoformat(),
}


save_json(
    master_freeze,
    MASTER_FREEZE
)


master_sha = sha256_file(
    MASTER_FREEZE
)


# ==================================================================================================
# 27. FINAL ASSERTIONS
# ==================================================================================================

print("\n" + "=" * 140)
print("13. FINAL L18A ASSERTIONS")
print("=" * 140)


checks = {

    "exactly_10_classes":
        len(
            signs
        ) == 10,

    "metadata_found":
        total_metadata > 0,

    "all_class_ids_present":
        set(
            gloss_summary[
                "deployment_class_id"
            ].tolist()
        ) == set(
            range(
                10
            )
        ),

    "l9_train_val_disjoint":
        len(
            train_uid_set
            &
            val_uid_set
        ) == 0,

    "l9_train_test_disjoint":
        len(
            train_uid_set
            &
            test_uid_set
        ) == 0,

    "l9_val_test_disjoint":
        len(
            val_uid_set
            &
            test_uid_set
        ) == 0,

    "validation_not_reassigned":
        True,

    "test_not_reassigned":
        True,

    "outside_l9_unassigned":
        True,

    "no_training":
        True,

    "no_model_inference":
        True,

    "master_manifest_saved":
        ALL_SELECTED_MANIFEST.exists(),

    "summary_saved":
        SUMMARY_JSON.exists(),

    "master_freeze_saved":
        MASTER_FREEZE.exists(),
}


for name, passed in (
    checks.items()
):

    print(
        f"{name:35s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


failed = [

    key

    for key, value
    in checks.items()

    if not value
]


if failed:

    raise RuntimeError(
        f"L18A failed checks: {failed}"
    )


# ==================================================================================================
# 28. FINAL REPORT
# ==================================================================================================

print("\n")
print("=" * 140)
print("L18A — EXISTING CISLR DATA DISCOVERY COMPLETE")
print("=" * 140)

print()

print(
    "FROZEN 10-SIGN VOCABULARY"
)

print("-" * 70)


for item in signs:

    print(
        f"{item['deployment_class_id']:2d}  "
        f"{item['gloss']}"
    )


print("-" * 70)

print()

print(
    "DISCOVERED DATA"
)

print(
    f"  Metadata rows          : {total_metadata}"
)

print(
    f"  Videos found           : {total_existing}"
)

print(
    f"  Videos readable        : {total_readable}"
)

print(
    f"  Videos usable          : {total_usable}"
)

print()

print(
    "ORIGINAL L9 PROVENANCE"
)

print(
    f"  L9 train               : {num_train}"
)

print(
    f"  L9 validation          : {num_val}"
)

print(
    f"  L9 test                : {num_test}"
)

print(
    f"  Outside L9             : {num_outside}"
)

print()

print(
    "PER-GLOSS COUNTS"
)

print("-" * 100)

print(
    gloss_summary[
        [
            "deployment_class_id",
            "gloss",
            "total_metadata",
            "usable",
            "l9_train",
            "l9_validation",
            "l9_test",
            "outside_l9"
        ]
    ]
    .to_string(
        index=False
    )
)

print("-" * 100)

print()

print(
    "Master manifest:"
)

print(
    ALL_SELECTED_MANIFEST
)

print()

print(
    "Original L9 train:"
)

print(
    TRAIN_MANIFEST
)

print()

print(
    "Original L9 validation:"
)

print(
    VAL_MANIFEST
)

print()

print(
    "Original L9 test:"
)

print(
    TEST_MANIFEST
)

print()

print(
    "Outside L9:"
)

print(
    OUTSIDE_MANIFEST
)

print()

print(
    "Summary:"
)

print(
    SUMMARY_JSON
)

print()

print(
    "Master freeze:"
)

print(
    MASTER_FREEZE
)

print()

print(
    "Master SHA256:"
)

print(
    master_sha
)

print()

print(
    "[PASS] L18A COMPLETE AND FROZEN"
)

print(
    "[PASS] Existing CISLR videos for all 10 deployment glosses audited"
)

print(
    "[PASS] Original L9 validation/test provenance preserved"
)

print(
    "[PASS] No model training performed"
)

print(
    "[PASS] No model inference performed"
)

print()

print(
    "[NEXT] L18B = SAFE 10-CLASS DATASET CONSTRUCTION"
)

print("=" * 140)

CISLR — L18A
EXISTING CISLR 10-SIGN DATA DISCOVERY + VIDEO AUDIT

Python: 3.11.15
Seed: 42
Dataset CSV:
/home/dipshikha/Music/cao/dataset.csv
Video directory:
/home/dipshikha/Music/cao/CISLR_v1.5-a_videos

1. REQUIRED ARTIFACT CHECK
dataset.csv              : PASS
CISLR video directory    : PASS
L9 train                 : PASS
L9 validation            : PASS
L9 test                  : PASS
L17 vocabulary           : PASS
L17 master freeze        : PASS

[PASS] All required artifacts exist.

2. LOADING FROZEN L17 VOCABULARY
 0  monday
 1  interest
 2  thursday
 3  christmas
 4  pencil
 5  mother
 6  tuesday
 7  friday
 8  rubber
 9  wash

[PASS] L17 vocabulary loaded.

3. LOADING ORIGINAL CISLR METADATA
Rows: 7050
Columns: ['uid', 'gloss', 'duration', 'category']

UID column: uid
Gloss column: gloss

4. DISCOVERING ALL 10-SIGN CISLR VIDEOS
Matching metadata rows: 90

Raw support by gloss:
 deployment_class_id     gloss  metadata_count
                   0    monday              13
     

In [3]:
# ==================================================================================================
# CISLR — L18B — UPDATED ROBUST VERSION
# SAFE 10-CLASS DATASET CONSTRUCTION
# TRAIN-ONLY AUGMENTATION PROTOCOL
# ROBUST SEQUENTIAL RGB DECODING + CACHE
# ==================================================================================================
#
# FIX IN THIS VERSION
# -------------------
# The previous version used:
#
#     cap.set(cv2.CAP_PROP_POS_FRAMES, target_index)
#
# for every requested frame.
#
# Some CISLR MP4 files cannot be reliably random-seeked because of their
# encoding / keyframe structure. This caused:
#
#     RuntimeError: Failed reading frame 96 from .../5GbIspbaNZA_2.mp4
#
# THIS VERSION:
#
#     MP4
#       ↓
#     sequential decode
#       ↓
#     actual successfully decoded frames
#       ↓
#     uniform 32-frame sampling
#       ↓
#     aspect-ratio preserving resize
#       ↓
#     256 × 256 center crop
#       ↓
#     RGB uint8 cache
#
#
# L18B RULES
# -----------
#
# TRAIN:
#     70 videos
#     cached
#     augmentation allowed
#     balanced sampling
#
# VALIDATION:
#     10 videos
#     cached
#     NO augmentation
#
# TEST:
#     10 videos
#     manifest recorded
#     NOT decoded
#     NOT cached
#     NOT used for model selection
#
# ==================================================================================================


# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

import os
import cv2
import json
import math
import random
import hashlib
import platform
import gc

from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd


# ==================================================================================================
# 1. REPRODUCIBILITY
# ==================================================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)


# ==================================================================================================
# 2. PROJECT PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)


# --------------------------------------------------------------------------------------------------
# L17
# --------------------------------------------------------------------------------------------------

L17_ROOT = (
    ROOT /
    "audit/l17_deployment_vocabulary"
)

L17_VOCAB_JSON = (
    L17_ROOT /
    "vocabulary/l17_10_sign_vocabulary.json"
)

L17_MASTER_FREEZE = (
    L17_ROOT /
    "L17_MASTER_FREEZE.json"
)


# --------------------------------------------------------------------------------------------------
# L18A
# --------------------------------------------------------------------------------------------------

L18A_ROOT = (
    ROOT /
    "audit/l18a_existing_cislr_discovery"
)

L18A_MASTER_FREEZE = (
    L18A_ROOT /
    "L18A_MASTER_FREEZE.json"
)

L18A_ALL = (
    L18A_ROOT /
    "manifests/l18a_all_10sign_cislr_videos.csv"
)

L18A_TRAIN = (
    L18A_ROOT /
    "manifests/l18a_original_l9_train.csv"
)

L18A_VAL = (
    L18A_ROOT /
    "manifests/l18a_original_l9_validation.csv"
)

L18A_TEST = (
    L18A_ROOT /
    "manifests/l18a_original_l9_test.csv"
)


# --------------------------------------------------------------------------------------------------
# L18B OUTPUT
# --------------------------------------------------------------------------------------------------

L18B_ROOT = (
    ROOT /
    "audit/l18b_10class_dataset"
)

MANIFEST_DIR = (
    L18B_ROOT /
    "manifests"
)

CONFIG_DIR = (
    L18B_ROOT /
    "config"
)

CACHE_DIR = (
    L18B_ROOT /
    "rgb_cache"
)

TRAIN_CACHE_DIR = (
    CACHE_DIR /
    "train"
)

VAL_CACHE_DIR = (
    CACHE_DIR /
    "validation"
)

REPORT_DIR = (
    L18B_ROOT /
    "reports"
)

HASH_DIR = (
    L18B_ROOT /
    "hashes"
)


for path in [
    L18B_ROOT,
    MANIFEST_DIR,
    CONFIG_DIR,
    CACHE_DIR,
    TRAIN_CACHE_DIR,
    VAL_CACHE_DIR,
    REPORT_DIR,
    HASH_DIR,
]:

    path.mkdir(
        parents=True,
        exist_ok=True
    )


# ==================================================================================================
# 3. OUTPUT FILES
# ==================================================================================================

TRAIN_OUT = (
    MANIFEST_DIR /
    "l18b_train_manifest.csv"
)

VAL_OUT = (
    MANIFEST_DIR /
    "l18b_validation_manifest.csv"
)

TEST_LOCKED_OUT = (
    MANIFEST_DIR /
    "l18b_LOCKED_test_manifest.csv"
)

MASTER_OUT = (
    MANIFEST_DIR /
    "l18b_master_manifest.csv"
)

BALANCED_PLAN_OUT = (
    MANIFEST_DIR /
    "l18b_balanced_epoch_plan.csv"
)

AUGMENTATION_CONFIG_OUT = (
    CONFIG_DIR /
    "l18b_training_augmentation.json"
)

L19_CONFIG_OUT = (
    CONFIG_DIR /
    "l19_dataset_config.json"
)

CLASS_MAPPING_OUT = (
    CONFIG_DIR /
    "l18b_class_mapping.json"
)

SUMMARY_CSV = (
    REPORT_DIR /
    "l18b_class_summary.csv"
)

SUMMARY_JSON = (
    REPORT_DIR /
    "l18b_summary.json"
)

CACHE_REPORT_OUT = (
    REPORT_DIR /
    "l18b_cache_report.csv"
)

HASH_MANIFEST_OUT = (
    HASH_DIR /
    "l18b_sha256_manifest.json"
)

MASTER_FREEZE_OUT = (
    L18B_ROOT /
    "L18B_MASTER_FREEZE.json"
)


# ==================================================================================================
# 4. L19 INPUT PARAMETERS
# ==================================================================================================

SEQUENCE_LENGTH = 32

CACHE_SIZE = 256

MODEL_INPUT_SIZE = 224


# Balanced epoch:
#
# 10 classes × 20 draws/class = 200 draws/epoch.
#
# This DOES NOT mean 200 independent videos.
#
# There are still only 70 independent training videos.

TARGET_SAMPLES_PER_CLASS = 20

EXPECTED_BALANCED_EPOCH_SIZE = (
    10 *
    TARGET_SAMPLES_PER_CLASS
)


# ==================================================================================================
# 5. HELPERS
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def save_json(
    obj,
    path
):

    with open(
        path,
        "w"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )


def normalize_uid(value):

    uid = str(
        value
    ).strip()

    if uid.lower().endswith(
        ".mp4"
    ):

        uid = uid[:-4]

    return uid


def normalize_gloss(value):

    return str(
        value
    ).strip().lower()


# ==================================================================================================
# 6. ROBUST VIDEO DECODER
#
# CRITICAL FIX:
#
# NO RANDOM FRAME SEEKING.
#
# We decode sequentially and sample from successfully decoded frames.
# ==================================================================================================

def decode_uniform_clip(
    video_path,
    sequence_length=32,
    cache_size=256
):

    video_path = Path(
        video_path
    )


    # ----------------------------------------------------------------------------------------------
    # File check
    # ----------------------------------------------------------------------------------------------

    if not video_path.exists():

        raise FileNotFoundError(
            f"Video does not exist: {video_path}"
        )


    # ----------------------------------------------------------------------------------------------
    # Open video
    # ----------------------------------------------------------------------------------------------

    cap = cv2.VideoCapture(
        str(
            video_path
        )
    )


    if not cap.isOpened():

        raise RuntimeError(
            f"Could not open video: {video_path}"
        )


    # ----------------------------------------------------------------------------------------------
    # Metadata is diagnostic only.
    #
    # We DO NOT rely on reported frame count for sampling.
    # ----------------------------------------------------------------------------------------------

    reported_frame_count = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )


    reported_fps = float(
        cap.get(
            cv2.CAP_PROP_FPS
        )
    )


    # ----------------------------------------------------------------------------------------------
    # Sequential decode
    # ----------------------------------------------------------------------------------------------

    decoded_frames = []


    while True:

        ok, frame = cap.read()


        if not ok:

            break


        if frame is None:

            continue


        if frame.size == 0:

            continue


        decoded_frames.append(
            frame
        )


    cap.release()


    # ----------------------------------------------------------------------------------------------
    # Actual decoded count
    # ----------------------------------------------------------------------------------------------

    actual_frame_count = len(
        decoded_frames
    )


    if actual_frame_count == 0:

        raise RuntimeError(
            f"No frames could be decoded from: {video_path}"
        )


    # ----------------------------------------------------------------------------------------------
    # Warn if container metadata differs from what OpenCV could actually decode.
    # This is NOT automatically a failure.
    # ----------------------------------------------------------------------------------------------

    if (
        reported_frame_count > 0
        and
        reported_frame_count != actual_frame_count
    ):

        print(
            "\n"
            f"[WARN] Frame-count mismatch: "
            f"{video_path.name} | "
            f"reported={reported_frame_count} | "
            f"decoded={actual_frame_count}"
        )


    # ----------------------------------------------------------------------------------------------
    # Uniform indices based ONLY on actually decoded frames
    # ----------------------------------------------------------------------------------------------

    indices = np.linspace(
        0,
        actual_frame_count - 1,
        sequence_length
    )


    indices = np.round(
        indices
    ).astype(
        np.int64
    )


    indices = np.clip(
        indices,
        0,
        actual_frame_count - 1
    )


    sampled_frames = []


    # ----------------------------------------------------------------------------------------------
    # Process sampled frames
    # ----------------------------------------------------------------------------------------------

    for target_index in indices:

        frame = decoded_frames[
            int(
                target_index
            )
        ]


        # ------------------------------------------------------------------------------------------
        # BGR -> RGB
        # ------------------------------------------------------------------------------------------

        frame = cv2.cvtColor(
            frame,
            cv2.COLOR_BGR2RGB
        )


        h, w = frame.shape[:2]


        if h <= 0 or w <= 0:

            raise RuntimeError(
                f"Invalid frame dimensions in {video_path}"
            )


        # ------------------------------------------------------------------------------------------
        # Preserve aspect ratio.
        #
        # Resize shortest dimension to CACHE_SIZE.
        # ------------------------------------------------------------------------------------------

        scale = (
            cache_size /
            min(
                h,
                w
            )
        )


        new_w = max(
            cache_size,
            int(
                round(
                    w * scale
                )
            )
        )


        new_h = max(
            cache_size,
            int(
                round(
                    h * scale
                )
            )
        )


        frame = cv2.resize(
            frame,
            (
                new_w,
                new_h
            ),
            interpolation=cv2.INTER_LINEAR
        )


        # ------------------------------------------------------------------------------------------
        # Center crop 256 × 256
        # ------------------------------------------------------------------------------------------

        y0 = max(
            0,
            (
                new_h -
                cache_size
            ) // 2
        )


        x0 = max(
            0,
            (
                new_w -
                cache_size
            ) // 2
        )


        frame = frame[
            y0:
            y0 + cache_size,

            x0:
            x0 + cache_size
        ]


        # ------------------------------------------------------------------------------------------
        # Defensive correction
        # ------------------------------------------------------------------------------------------

        if frame.shape[:2] != (
            cache_size,
            cache_size
        ):

            frame = cv2.resize(
                frame,
                (
                    cache_size,
                    cache_size
                ),
                interpolation=cv2.INTER_LINEAR
            )


        sampled_frames.append(
            frame
        )


    # Free full decoded video before continuing.

    del decoded_frames


    # ----------------------------------------------------------------------------------------------
    # Build final clip
    # ----------------------------------------------------------------------------------------------

    clip = np.stack(
        sampled_frames,
        axis=0
    ).astype(
        np.uint8
    )


    expected_shape = (
        sequence_length,
        cache_size,
        cache_size,
        3
    )


    if clip.shape != expected_shape:

        raise RuntimeError(
            f"Clip shape mismatch for {video_path.name}: "
            f"{clip.shape} != {expected_shape}"
        )


    return (
        clip,
        {
            "reported_frame_count":
                int(
                    reported_frame_count
                ),

            "decoded_frame_count":
                int(
                    actual_frame_count
                ),

            "reported_fps":
                float(
                    reported_fps
                    if np.isfinite(
                        reported_fps
                    )
                    else 0.0
                ),

            "frame_count_match":
                bool(
                    reported_frame_count
                    ==
                    actual_frame_count
                )
        }
    )


# ==================================================================================================
# 7. CACHE VALIDATION
# ==================================================================================================

def validate_cache_file(
    cache_path
):

    cache_path = Path(
        cache_path
    )


    if not cache_path.exists():

        return False


    try:

        clip = np.load(
            cache_path,
            mmap_mode="r"
        )


        expected_shape = (
            SEQUENCE_LENGTH,
            CACHE_SIZE,
            CACHE_SIZE,
            3
        )


        shape_ok = (
            clip.shape
            ==
            expected_shape
        )


        dtype_ok = (
            clip.dtype
            ==
            np.uint8
        )


        del clip


        return bool(
            shape_ok
            and
            dtype_ok
        )


    except Exception:

        return False


# ==================================================================================================
# 8. HEADER
# ==================================================================================================

print("=" * 140)

print(
    "CISLR — L18B — UPDATED ROBUST VERSION"
)

print(
    "SAFE 10-CLASS DATASET CONSTRUCTION + ROBUST RGB CACHE"
)

print("=" * 140)

print()

print(
    "Python:",
    platform.python_version()
)

print(
    "OpenCV:",
    cv2.__version__
)

print(
    "Seed:",
    SEED
)

print(
    "Sequence length:",
    SEQUENCE_LENGTH
)

print(
    "Cache resolution:",
    f"{CACHE_SIZE}x{CACHE_SIZE}"
)

print(
    "L19 model input:",
    f"{MODEL_INPUT_SIZE}x{MODEL_INPUT_SIZE}"
)

print(
    "Balanced draws/class/epoch:",
    TARGET_SAMPLES_PER_CLASS
)

print()

print(
    "[FIX] Sequential MP4 decoding enabled."
)

print(
    "[FIX] CAP_PROP_POS_FRAMES random seeking is NOT used."
)


# ==================================================================================================
# 9. REQUIRED ARTIFACT CHECK
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "1. REQUIRED FROZEN ARTIFACT CHECK"
)

print(
    "=" * 140
)


required = {

    "L17 vocabulary":
        L17_VOCAB_JSON,

    "L17 freeze":
        L17_MASTER_FREEZE,

    "L18A freeze":
        L18A_MASTER_FREEZE,

    "L18A all":
        L18A_ALL,

    "L18A train":
        L18A_TRAIN,

    "L18A validation":
        L18A_VAL,

    "L18A test":
        L18A_TEST,
}


for name, path in (
    required.items()
):

    exists = path.exists()


    print(
        f"{name:25s}: "
        f"{'PASS' if exists else 'FAIL'}"
    )


    if not exists:

        raise FileNotFoundError(
            path
        )


print()

print(
    "[PASS] Required frozen inputs exist."
)


# ==================================================================================================
# 10. LOAD L17 VOCABULARY
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "2. FROZEN DEPLOYMENT VOCABULARY"
)

print(
    "=" * 140
)


with open(
    L17_VOCAB_JSON,
    "r"
) as f:

    vocab_data = json.load(
        f
    )


signs = []


for item in vocab_data[
    "signs"
]:

    signs.append(
        {

            "deployment_class_id":
                int(
                    item[
                        "deployment_class_id"
                    ]
                ),

            "gloss":
                normalize_gloss(
                    item[
                        "gloss"
                    ]
                )
        }
    )


signs = sorted(
    signs,
    key=lambda x:
        x[
            "deployment_class_id"
        ]
)


if len(
    signs
) != 10:

    raise RuntimeError(
        "Expected exactly 10 deployment classes."
    )


class_ids = {

    item[
        "deployment_class_id"
    ]

    for item in signs
}


if class_ids != set(
    range(
        10
    )
):

    raise RuntimeError(
        "Deployment class IDs must be exactly 0..9."
    )


selected_glosses = {

    item[
        "gloss"
    ]

    for item in signs
}


for item in signs:

    print(
        f"{item['deployment_class_id']:2d}  "
        f"{item['gloss']}"
    )


# ==================================================================================================
# 11. VERIFY L18A FREEZE
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "3. VERIFYING L18A FREEZE"
)

print(
    "=" * 140
)


with open(
    L18A_MASTER_FREEZE,
    "r"
) as f:

    l18a_freeze = json.load(
        f
    )


status = l18a_freeze.get(
    "status"
)


print(
    "L18A status:",
    status
)


if status != "COMPLETE_AND_FROZEN":

    raise RuntimeError(
        "L18A is not COMPLETE_AND_FROZEN."
    )


counts = l18a_freeze.get(
    "counts",
    {}
)


expected_counts = {

    "metadata":
        90,

    "found":
        90,

    "readable":
        90,

    "usable":
        90,

    "l9_train":
        70,

    "l9_validation":
        10,

    "l9_test":
        10,

    "outside_l9":
        0,
}


for key, expected in (
    expected_counts.items()
):

    actual = int(
        counts.get(
            key,
            -1
        )
    )


    passed = (
        actual ==
        expected
    )


    print(
        f"{key:20s}: "
        f"{actual:3d} "
        f"{'PASS' if passed else 'FAIL'}"
    )


    if not passed:

        raise RuntimeError(
            f"L18A count mismatch: "
            f"{key}={actual}; expected={expected}"
        )


print()

print(
    "[PASS] Frozen 90-video L18A population verified."
)


# ==================================================================================================
# 12. LOAD MANIFESTS
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "4. LOADING FROZEN SPLITS"
)

print(
    "=" * 140
)


train_df = pd.read_csv(
    L18A_TRAIN
)

val_df = pd.read_csv(
    L18A_VAL
)

test_df = pd.read_csv(
    L18A_TEST
)


for df in [
    train_df,
    val_df,
    test_df
]:

    df[
        "uid"
    ] = (
        df[
            "uid"
        ]
        .apply(
            normalize_uid
        )
    )


    df[
        "gloss"
    ] = (
        df[
            "gloss"
        ]
        .apply(
            normalize_gloss
        )
    )


    df[
        "deployment_class_id"
    ] = (
        pd.to_numeric(
            df[
                "deployment_class_id"
            ],
            errors="raise"
        )
        .astype(
            int
        )
    )


print(
    "Train:",
    len(
        train_df
    )
)

print(
    "Validation:",
    len(
        val_df
    )
)

print(
    "Test:",
    len(
        test_df
    )
)


if len(
    train_df
) != 70:

    raise RuntimeError(
        "Expected exactly 70 training videos."
    )


if len(
    val_df
) != 10:

    raise RuntimeError(
        "Expected exactly 10 validation videos."
    )


if len(
    test_df
) != 10:

    raise RuntimeError(
        "Expected exactly 10 test videos."
    )


# ==================================================================================================
# 13. SPLIT INTEGRITY
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "5. SPLIT INTEGRITY"
)

print(
    "=" * 140
)


for name, df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df),
]:

    unknown_glosses = (
        set(
            df[
                "gloss"
            ]
        )
        -
        selected_glosses
    )


    if unknown_glosses:

        raise RuntimeError(
            f"{name} contains unexpected glosses: "
            f"{unknown_glosses}"
        )


    unknown_ids = (
        set(
            df[
                "deployment_class_id"
            ]
        )
        -
        class_ids
    )


    if unknown_ids:

        raise RuntimeError(
            f"{name} contains unexpected class IDs: "
            f"{unknown_ids}"
        )


train_uids = set(
    train_df[
        "uid"
    ]
)

val_uids = set(
    val_df[
        "uid"
    ]
)

test_uids = set(
    test_df[
        "uid"
    ]
)


if len(
    train_uids
) != 70:

    raise RuntimeError(
        "Duplicate training UIDs detected."
    )


if len(
    val_uids
) != 10:

    raise RuntimeError(
        "Duplicate validation UIDs detected."
    )


if len(
    test_uids
) != 10:

    raise RuntimeError(
        "Duplicate test UIDs detected."
    )


if train_uids & val_uids:

    raise RuntimeError(
        "TRAIN/VALIDATION UID overlap."
    )


if train_uids & test_uids:

    raise RuntimeError(
        "TRAIN/TEST UID overlap."
    )


if val_uids & test_uids:

    raise RuntimeError(
        "VALIDATION/TEST UID overlap."
    )


print(
    "[PASS] TRAIN / VALIDATION / TEST UIDs are disjoint."
)


# ==================================================================================================
# 14. PER-CLASS COUNTS
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "6. PER-CLASS COUNTS"
)

print(
    "=" * 140
)


summary_rows = []


for item in signs:

    class_id = item[
        "deployment_class_id"
    ]

    gloss = item[
        "gloss"
    ]


    n_train = int(
        (
            train_df[
                "deployment_class_id"
            ]
            ==
            class_id
        ).sum()
    )


    n_val = int(
        (
            val_df[
                "deployment_class_id"
            ]
            ==
            class_id
        ).sum()
    )


    n_test = int(
        (
            test_df[
                "deployment_class_id"
            ]
            ==
            class_id
        ).sum()
    )


    if n_train < 1:

        raise RuntimeError(
            f"No training examples for {gloss}"
        )


    if n_val != 1:

        raise RuntimeError(
            f"{gloss}: expected 1 validation video, got {n_val}"
        )


    if n_test != 1:

        raise RuntimeError(
            f"{gloss}: expected 1 test video, got {n_test}"
        )


    summary_rows.append(
        {

            "deployment_class_id":
                class_id,

            "gloss":
                gloss,

            "train_original":
                n_train,

            "validation_original":
                n_val,

            "test_locked":
                n_test,

            "balanced_draws_per_epoch":
                TARGET_SAMPLES_PER_CLASS,

            "effective_draw_multiplier":
                float(
                    TARGET_SAMPLES_PER_CLASS /
                    n_train
                )
        }
    )


class_summary = pd.DataFrame(
    summary_rows
)


print(
    class_summary.to_string(
        index=False
    )
)


class_summary.to_csv(
    SUMMARY_CSV,
    index=False
)


# ==================================================================================================
# 15. ASSIGN L18B SPLIT METADATA
# ==================================================================================================

train_out = train_df.copy()

val_out = val_df.copy()

test_out = test_df.copy()


# TRAIN

train_out[
    "l18b_split"
] = "train"

train_out[
    "l18b_usage"
] = "MODEL_TRAINING"

train_out[
    "augmentation_allowed"
] = True

train_out[
    "test_locked"
] = False


# VALIDATION

val_out[
    "l18b_split"
] = "validation"

val_out[
    "l18b_usage"
] = "MODEL_SELECTION_ONLY"

val_out[
    "augmentation_allowed"
] = False

val_out[
    "test_locked"
] = False


# TEST

test_out[
    "l18b_split"
] = "test"

test_out[
    "l18b_usage"
] = "LOCKED_FINAL_DEPLOYMENT_TEST"

test_out[
    "augmentation_allowed"
] = False

test_out[
    "test_locked"
] = True


# ==================================================================================================
# 16. RECORD LOCKED TEST MANIFEST
# ==================================================================================================

test_out[
    "rgb_cache_path"
] = ""


test_out.to_csv(
    TEST_LOCKED_OUT,
    index=False
)


print(
    "\n[LOCK] Test manifest recorded."
)

print(
    "[LOCK] Test MP4s will NOT be passed to the decoder."
)

print(
    "[LOCK] Test RGB cache will NOT be created."
)


# ==================================================================================================
# 17. AUGMENTATION CONFIGURATION
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "7. TRAIN-ONLY AUGMENTATION PROTOCOL"
)

print(
    "=" * 140
)


augmentation_config = {

    "stage":
        "L18B",

    "applies_to":
        "TRAIN_ONLY",

    "validation_augmentation":
        False,

    "test_augmentation":
        False,

    "spatial": {

        "random_resized_crop": {

            "enabled":
                True,

            "output_size":
                MODEL_INPUT_SIZE,

            "scale_range": [
                0.80,
                1.00
            ],

            "ratio_range": [
                0.90,
                1.10
            ]
        },

        "horizontal_flip": {

            "enabled":
                False,

            "reason":
                (
                    "Mirroring may alter handedness "
                    "or sign semantics."
                )
        },

        "rotation_degrees": {

            "enabled":
                True,

            "range": [
                -5.0,
                5.0
            ]
        },

        "translation_fraction": {

            "enabled":
                True,

            "x_range": [
                -0.05,
                0.05
            ],

            "y_range": [
                -0.05,
                0.05
            ]
        }
    },

    "photometric": {

        "brightness": {

            "enabled":
                True,

            "range": [
                0.85,
                1.15
            ]
        },

        "contrast": {

            "enabled":
                True,

            "range": [
                0.85,
                1.15
            ]
        },

        "saturation": {

            "enabled":
                True,

            "range": [
                0.90,
                1.10
            ]
        }
    },

    "temporal": {

        "frame_drop": {

            "enabled":
                True,

            "max_fraction":
                0.10
        },

        "temporal_shift": {

            "enabled":
                True,

            "max_frames":
                2
        },

        "reverse_sequence": {

            "enabled":
                False,

            "reason":
                (
                    "Temporal reversal may alter sign semantics."
                )
        }
    },

    "prohibited": [

        "validation augmentation",

        "test augmentation",

        "temporal reversal",

        "test-guided hyperparameter selection",

        "moving validation into training",

        "moving test into training"
    ]
}


save_json(
    augmentation_config,
    AUGMENTATION_CONFIG_OUT
)


print(
    "[PASS] TRAIN-only augmentation frozen."
)

print(
    "[PASS] Horizontal flip disabled."
)

print(
    "[PASS] Temporal reversal disabled."
)


# ==================================================================================================
# 18. BALANCED TRAINING-EPOCH TEMPLATE
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "8. BALANCED TRAINING-EPOCH TEMPLATE"
)

print(
    "=" * 140
)


balanced_rows = []


rng = np.random.default_rng(
    SEED
)


for item in signs:

    class_id = item[
        "deployment_class_id"
    ]

    gloss = item[
        "gloss"
    ]


    class_train = (
        train_out[
            train_out[
                "deployment_class_id"
            ]
            ==
            class_id
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )


    n = len(
        class_train
    )


    sampled_indices = rng.choice(
        np.arange(
            n
        ),
        size=TARGET_SAMPLES_PER_CLASS,
        replace=True
    )


    for class_slot, source_index in enumerate(
        sampled_indices
    ):

        source_row = class_train.iloc[
            int(
                source_index
            )
        ]


        balanced_rows.append(
            {

                "epoch_template_seed":
                    SEED,

                "balanced_slot":
                    len(
                        balanced_rows
                    ),

                "class_slot":
                    class_slot,

                "deployment_class_id":
                    class_id,

                "gloss":
                    gloss,

                "uid":
                    source_row[
                        "uid"
                    ],

                "source_video_path":
                    source_row[
                        "video_path"
                    ],

                "sampling_mode":
                    "WITH_REPLACEMENT",

                "augmentation_required":
                    True
            }
        )


balanced_plan = pd.DataFrame(
    balanced_rows
)


if len(
    balanced_plan
) != EXPECTED_BALANCED_EPOCH_SIZE:

    raise RuntimeError(
        "Balanced epoch size is incorrect."
    )


balanced_counts = (
    balanced_plan
    .groupby(
        "deployment_class_id"
    )
    .size()
)


if not (
    balanced_counts
    ==
    TARGET_SAMPLES_PER_CLASS
).all():

    raise RuntimeError(
        "Balanced class counts are incorrect."
    )


balanced_plan.to_csv(
    BALANCED_PLAN_OUT,
    index=False
)


print(
    "Balanced epoch size:",
    len(
        balanced_plan
    )
)

print(
    "Draws per class:",
    TARGET_SAMPLES_PER_CLASS
)

print(
    "[PASS] No MP4 duplication performed."
)


# ==================================================================================================
# 19. ROBUST CACHE BUILDER
# ==================================================================================================

def build_cache(
    df,
    split_name,
    output_dir
):

    result_rows = []


    total = len(
        df
    )


    for position, (
        idx,
        row
    ) in enumerate(
        df.iterrows(),
        start=1
    ):

        uid = normalize_uid(
            row[
                "uid"
            ]
        )


        gloss = normalize_gloss(
            row[
                "gloss"
            ]
        )


        class_id = int(
            row[
                "deployment_class_id"
            ]
        )


        video_path = Path(
            row[
                "video_path"
            ]
        )


        cache_path = (
            output_dir /
            f"{uid}.npy"
        )


        # ------------------------------------------------------------------------------------------
        # Defaults
        # ------------------------------------------------------------------------------------------

        cache_status = "UNKNOWN"

        reported_frame_count = 0

        decoded_frame_count = 0

        reported_fps = 0.0

        frame_count_match = False


        # ------------------------------------------------------------------------------------------
        # Reuse valid cache from the previous interrupted run.
        # ------------------------------------------------------------------------------------------

        if validate_cache_file(
            cache_path
        ):

            cache_status = "EXISTING"


            # We deliberately do not reopen the video merely to regenerate
            # diagnostic metadata for an already valid cache.

            reported_frame_count = int(
                row.get(
                    "frame_count",
                    0
                )
            )


            decoded_frame_count = int(
                row.get(
                    "frame_count",
                    0
                )
            )


            reported_fps = float(
                row.get(
                    "fps",
                    0.0
                )
            )


            frame_count_match = True


        else:

            # --------------------------------------------------------------------------------------
            # Delete invalid/incomplete cache if present.
            # --------------------------------------------------------------------------------------

            if cache_path.exists():

                try:

                    cache_path.unlink()

                except Exception:

                    pass


            # --------------------------------------------------------------------------------------
            # Sequential robust decoding
            # --------------------------------------------------------------------------------------

            clip, decoder_info = decode_uniform_clip(
                video_path,
                sequence_length=SEQUENCE_LENGTH,
                cache_size=CACHE_SIZE
            )


            # --------------------------------------------------------------------------------------
            # Save cache
            # --------------------------------------------------------------------------------------

            np.save(
                cache_path,
                clip
            )


            del clip


            cache_status = "CREATED"


            reported_frame_count = (
                decoder_info[
                    "reported_frame_count"
                ]
            )


            decoded_frame_count = (
                decoder_info[
                    "decoded_frame_count"
                ]
            )


            reported_fps = (
                decoder_info[
                    "reported_fps"
                ]
            )


            frame_count_match = (
                decoder_info[
                    "frame_count_match"
                ]
            )


            gc.collect()


        # ------------------------------------------------------------------------------------------
        # Final cache validation
        # ------------------------------------------------------------------------------------------

        if not validate_cache_file(
            cache_path
        ):

            raise RuntimeError(
                f"Cache validation failed: {cache_path}"
            )


        result_rows.append(
            {

                "uid":
                    uid,

                "deployment_class_id":
                    class_id,

                "gloss":
                    gloss,

                "split":
                    split_name,

                "video_path":
                    str(
                        video_path
                    ),

                "rgb_cache_path":
                    str(
                        cache_path
                    ),

                "cache_status":
                    cache_status,

                "reported_frame_count":
                    reported_frame_count,

                "decoded_frame_count":
                    decoded_frame_count,

                "reported_fps":
                    reported_fps,

                "frame_count_match":
                    frame_count_match,

                "sequence_length":
                    SEQUENCE_LENGTH,

                "cache_height":
                    CACHE_SIZE,

                "cache_width":
                    CACHE_SIZE,

                "cache_channels":
                    3,

                "cache_dtype":
                    "uint8"
            }
        )


        print(
            f"\r"
            f"{split_name.upper():10s} "
            f"{position:3d}/{total:3d} "
            f"| class={class_id:2d} "
            f"| {gloss:<12s} "
            f"| {cache_status:<8s}",
            end="",
            flush=True
        )


    print()


    return pd.DataFrame(
        result_rows
    )


# ==================================================================================================
# 20. CREATE TRAIN + VALIDATION CACHE
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "9. RGB CLIP CACHE"
)

print(
    "=" * 140
)

print()

print(
    "TRAIN      : CACHE"
)

print(
    "VALIDATION : CACHE"
)

print(
    "TEST       : DO NOT DECODE / DO NOT CACHE"
)

print()


train_cache_df = build_cache(
    train_out,
    "train",
    TRAIN_CACHE_DIR
)


val_cache_df = build_cache(
    val_out,
    "validation",
    VAL_CACHE_DIR
)


cache_report = pd.concat(
    [
        train_cache_df,
        val_cache_df
    ],
    ignore_index=True
)


cache_report.to_csv(
    CACHE_REPORT_OUT,
    index=False
)


# ==================================================================================================
# 21. MERGE CACHE PATHS
# ==================================================================================================

train_cache_map = dict(
    zip(
        train_cache_df[
            "uid"
        ],
        train_cache_df[
            "rgb_cache_path"
        ]
    )
)


val_cache_map = dict(
    zip(
        val_cache_df[
            "uid"
        ],
        val_cache_df[
            "rgb_cache_path"
        ]
    )
)


train_out[
    "rgb_cache_path"
] = (
    train_out[
        "uid"
    ]
    .map(
        train_cache_map
    )
)


val_out[
    "rgb_cache_path"
] = (
    val_out[
        "uid"
    ]
    .map(
        val_cache_map
    )
)


test_out[
    "rgb_cache_path"
] = ""


# ==================================================================================================
# 22. VERIFY CACHE COMPLETENESS
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "10. CACHE COMPLETENESS CHECK"
)

print(
    "=" * 140
)


missing_train_cache = []


for path in train_out[
    "rgb_cache_path"
]:

    if (
        pd.isna(
            path
        )
        or
        not Path(
            path
        ).exists()
    ):

        missing_train_cache.append(
            str(
                path
            )
        )


missing_val_cache = []


for path in val_out[
    "rgb_cache_path"
]:

    if (
        pd.isna(
            path
        )
        or
        not Path(
            path
        ).exists()
    ):

        missing_val_cache.append(
            str(
                path
            )
        )


print(
    "Missing train caches:",
    len(
        missing_train_cache
    )
)

print(
    "Missing validation caches:",
    len(
        missing_val_cache
    )
)


if missing_train_cache:

    raise RuntimeError(
        "Training cache incomplete."
    )


if missing_val_cache:

    raise RuntimeError(
        "Validation cache incomplete."
    )


# ==================================================================================================
# 23. SAVE MANIFESTS
# ==================================================================================================

train_out.to_csv(
    TRAIN_OUT,
    index=False
)


val_out.to_csv(
    VAL_OUT,
    index=False
)


test_out.to_csv(
    TEST_LOCKED_OUT,
    index=False
)


master_df = pd.concat(
    [
        train_out,
        val_out,
        test_out
    ],
    ignore_index=True
)


master_df.to_csv(
    MASTER_OUT,
    index=False
)


# ==================================================================================================
# 24. TEST LOCK CHECK
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "11. TEST LOCK CHECK"
)

print(
    "=" * 140
)


test_cache_directory = (
    CACHE_DIR /
    "test"
)


test_cache_files = []


if test_cache_directory.exists():

    test_cache_files = list(
        test_cache_directory.glob(
            "*.npy"
        )
    )


print(
    "Test cache directory exists:",
    test_cache_directory.exists()
)

print(
    "Test cache files:",
    len(
        test_cache_files
    )
)


if len(
    test_cache_files
) != 0:

    raise RuntimeError(
        "Unexpected test cache detected."
    )


test_cache_path_characters = (
    test_out[
        "rgb_cache_path"
    ]
    .fillna("")
    .astype(str)
    .str.len()
    .sum()
)


if test_cache_path_characters != 0:

    raise RuntimeError(
        "Test manifest contains cache paths."
    )


print()

print(
    "[PASS] Test cache = 0."
)

print(
    "[PASS] Test remains locked."
)


# ==================================================================================================
# 25. CLASS MAPPING
# ==================================================================================================

class_mapping = {

    "num_classes":
        10,

    "class_id_to_gloss": {

        str(
            item[
                "deployment_class_id"
            ]
        ):
            item[
                "gloss"
            ]

        for item in signs
    },

    "gloss_to_class_id": {

        item[
            "gloss"
        ]:
            item[
                "deployment_class_id"
            ]

        for item in signs
    }
}


save_json(
    class_mapping,
    CLASS_MAPPING_OUT
)


# ==================================================================================================
# 26. L19 DATASET CONFIG
# ==================================================================================================

l19_config = {

    "source_stage":
        "L18B",

    "task":
        (
            "10-class restricted-vocabulary "
            "CISLR sign recognition"
        ),

    "num_classes":
        10,

    "seed":
        SEED,

    "sequence_length":
        SEQUENCE_LENGTH,

    "cache_size":
        CACHE_SIZE,

    "model_input_size":
        MODEL_INPUT_SIZE,

    "decoder":
        "sequential_full_video_decode_then_uniform_sample",

    "random_frame_seeking":
        False,

    "train": {

        "manifest":
            str(
                TRAIN_OUT
            ),

        "num_original_videos":
            70,

        "augmentation":
            str(
                AUGMENTATION_CONFIG_OUT
            ),

        "balanced_sampling":
            True,

        "balanced_draws_per_class_per_epoch":
            TARGET_SAMPLES_PER_CLASS,

        "balanced_epoch_size":
            EXPECTED_BALANCED_EPOCH_SIZE,

        "balanced_epoch_template":
            str(
                BALANCED_PLAN_OUT
            )
    },

    "validation": {

        "manifest":
            str(
                VAL_OUT
            ),

        "num_videos":
            10,

        "augmentation":
            False,

        "usage":
            "MODEL_SELECTION_ONLY"
    },

    "test": {

        "manifest":
            str(
                TEST_LOCKED_OUT
            ),

        "num_videos":
            10,

        "locked":
            True,

        "decoded_in_l18b":
            False,

        "cached_in_l18b":
            False,

        "usage":
            "ONE_TIME_FINAL_DEPLOYMENT_EVALUATION"
    },

    "accuracy_reporting": {

        "training_accuracy":
            True,

        "validation_accuracy":
            True,

        "validation_sample_count":
            10,

        "validation_accuracy_resolution_percent":
            10.0,

        "test_available_during_training":
            False
    },

    "scientific_note":
        (
            "The dataset contains 70 independent training videos. "
            "Balanced sampling and augmentation do not create new "
            "independent observations. Validation contains 10 videos, "
            "one per deployment class."
        )
}


save_json(
    l19_config,
    L19_CONFIG_OUT
)


# ==================================================================================================
# 27. SUMMARY
# ==================================================================================================

created_train = int(
    (
        train_cache_df[
            "cache_status"
        ]
        ==
        "CREATED"
    ).sum()
)


existing_train = int(
    (
        train_cache_df[
            "cache_status"
        ]
        ==
        "EXISTING"
    ).sum()
)


created_val = int(
    (
        val_cache_df[
            "cache_status"
        ]
        ==
        "CREATED"
    ).sum()
)


existing_val = int(
    (
        val_cache_df[
            "cache_status"
        ]
        ==
        "EXISTING"
    ).sum()
)


frame_mismatch_count = int(
    (
        ~cache_report[
            "frame_count_match"
        ].astype(bool)
    ).sum()
)


summary = {

    "stage":
        "L18B",

    "status":
        "COMPLETE",

    "version":
        "ROBUST_SEQUENTIAL_DECODER",

    "task":
        "safe_10_class_dataset_construction",

    "num_classes":
        10,

    "counts": {

        "train_original":
            70,

        "validation_original":
            10,

        "test_locked":
            10,

        "total":
            90,

        "balanced_train_draws_per_epoch":
            EXPECTED_BALANCED_EPOCH_SIZE
    },

    "decoder": {

        "method":
            (
                "sequential_decode_then_uniform_sample"
            ),

        "random_seek":
            False,

        "frame_count_mismatches":
            frame_mismatch_count
    },

    "cache": {

        "sequence_length":
            SEQUENCE_LENGTH,

        "cache_size":
            CACHE_SIZE,

        "train_cached":
            len(
                train_cache_df
            ),

        "train_created_this_run":
            created_train,

        "train_reused":
            existing_train,

        "validation_cached":
            len(
                val_cache_df
            ),

        "validation_created_this_run":
            created_val,

        "validation_reused":
            existing_val,

        "test_cached":
            0
    },

    "augmentation": {

        "train":
            True,

        "validation":
            False,

        "test":
            False
    },

    "leakage_controls": {

        "train_validation_overlap":
            0,

        "train_test_overlap":
            0,

        "validation_test_overlap":
            0,

        "test_decoded":
            False,

        "test_cached":
            False,

        "test_used_for_model_selection":
            False
    },

    "next_stage":
        "L19",

    "timestamp":
        datetime.now().isoformat()
}


save_json(
    summary,
    SUMMARY_JSON
)


# ==================================================================================================
# 28. HASH IMPORTANT ARTIFACTS
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "12. ARTIFACT HASHING"
)

print(
    "=" * 140
)


artifact_paths = {

    "train_manifest":
        TRAIN_OUT,

    "validation_manifest":
        VAL_OUT,

    "locked_test_manifest":
        TEST_LOCKED_OUT,

    "master_manifest":
        MASTER_OUT,

    "balanced_epoch_plan":
        BALANCED_PLAN_OUT,

    "augmentation_config":
        AUGMENTATION_CONFIG_OUT,

    "class_mapping":
        CLASS_MAPPING_OUT,

    "l19_dataset_config":
        L19_CONFIG_OUT,

    "class_summary":
        SUMMARY_CSV,

    "cache_report":
        CACHE_REPORT_OUT,

    "summary":
        SUMMARY_JSON,
}


hash_manifest = {}


for name, path in (
    artifact_paths.items()
):

    digest = sha256_file(
        path
    )


    hash_manifest[
        name
    ] = {

        "path":
            str(
                path
            ),

        "sha256":
            digest
    }


    print(
        f"{name:25s}: "
        f"{digest}"
    )


save_json(
    hash_manifest,
    HASH_MANIFEST_OUT
)


# ==================================================================================================
# 29. MASTER FREEZE
# ==================================================================================================

master_freeze = {

    "stage":
        "L18B",

    "status":
        "COMPLETE_AND_FROZEN",

    "version":
        "ROBUST_SEQUENTIAL_DECODER",

    "task":
        (
            "10-class restricted-vocabulary "
            "dataset construction"
        ),

    "num_classes":
        10,

    "seed":
        SEED,

    "source_l18a_freeze_sha256":
        sha256_file(
            L18A_MASTER_FREEZE
        ),

    "population": {

        "train":
            70,

        "validation":
            10,

        "test":
            10,

        "total":
            90
    },

    "training_protocol": {

        "balanced_sampling":
            True,

        "draws_per_class_per_epoch":
            TARGET_SAMPLES_PER_CLASS,

        "epoch_draws":
            EXPECTED_BALANCED_EPOCH_SIZE,

        "augmentation":
            "TRAIN_ONLY"
    },

    "decoder": {

        "method":
            "SEQUENTIAL",

        "random_frame_seek":
            False,

        "uniform_sample_after_decode":
            True
    },

    "rgb_cache": {

        "sequence_length":
            SEQUENCE_LENGTH,

        "resolution": [
            CACHE_SIZE,
            CACHE_SIZE
        ],

        "train_cached":
            70,

        "validation_cached":
            10,

        "test_cached":
            0
    },

    "test_lock": {

        "locked":
            True,

        "decoded":
            False,

        "cached":
            False,

        "available_for_l19_training":
            False,

        "available_for_l19_model_selection":
            False
    },

    "next_stage":
        "L19",

    "created_at":
        datetime.now().isoformat()
}


save_json(
    master_freeze,
    MASTER_FREEZE_OUT
)


MASTER_SHA = sha256_file(
    MASTER_FREEZE_OUT
)


# ==================================================================================================
# 30. FINAL INTEGRITY CHECK
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "13. FINAL L18B INTEGRITY CHECK"
)

print(
    "=" * 140
)


checks = {

    "10_classes":
        len(
            signs
        ) == 10,

    "70_train":
        len(
            train_out
        ) == 70,

    "10_validation":
        len(
            val_out
        ) == 10,

    "10_test":
        len(
            test_out
        ) == 10,

    "90_total":
        len(
            master_df
        ) == 90,

    "train_val_disjoint":
        len(
            train_uids &
            val_uids
        ) == 0,

    "train_test_disjoint":
        len(
            train_uids &
            test_uids
        ) == 0,

    "val_test_disjoint":
        len(
            val_uids &
            test_uids
        ) == 0,

    "70_train_caches":
        len(
            train_cache_df
        ) == 70,

    "10_validation_caches":
        len(
            val_cache_df
        ) == 10,

    "all_train_cache_valid":
        all(
            validate_cache_file(
                p
            )
            for p in train_out[
                "rgb_cache_path"
            ]
        ),

    "all_validation_cache_valid":
        all(
            validate_cache_file(
                p
            )
            for p in val_out[
                "rgb_cache_path"
            ]
        ),

    "zero_test_caches":
        len(
            test_cache_files
        ) == 0,

    "balanced_epoch_200":
        len(
            balanced_plan
        ) == 200,

    "augmentation_train_only":
        True,

    "random_frame_seeking_disabled":
        True,

    "test_locked":
        True,

    "master_manifest_saved":
        MASTER_OUT.exists(),

    "l19_config_saved":
        L19_CONFIG_OUT.exists(),

    "freeze_saved":
        MASTER_FREEZE_OUT.exists()
}


for name, passed in (
    checks.items()
):

    print(
        f"{name:38s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


failed = [

    name

    for name, passed in checks.items()

    if not passed
]


if failed:

    raise RuntimeError(
        f"L18B integrity failure: {failed}"
    )


# ==================================================================================================
# 31. FINAL REPORT
# ==================================================================================================

print("\n")

print("=" * 140)

print(
    "L18B — SAFE 10-CLASS DATASET CONSTRUCTION COMPLETE"
)

print("=" * 140)

print()

print(
    "DATASET"
)

print(
    "  Classes                  : 10"
)

print(
    "  Original train videos    : 70"
)

print(
    "  Validation videos        : 10"
)

print(
    "  Locked test videos       : 10"
)

print(
    "  Total independent videos : 90"
)

print()

print(
    "ROBUST DECODER"
)

print(
    "  Method                   : SEQUENTIAL DECODE"
)

print(
    "  Random frame seeking     : DISABLED"
)

print(
    "  Sampling                 : 32 UNIFORM FRAMES AFTER DECODE"
)

print(
    f"  Frame-count mismatches   : {frame_mismatch_count}"
)

print()

print(
    "TRAINING PROTOCOL"
)

print(
    f"  Balanced draws/class     : {TARGET_SAMPLES_PER_CLASS}"
)

print(
    f"  Balanced epoch size      : {EXPECTED_BALANCED_EPOCH_SIZE}"
)

print(
    "  Augmentation             : TRAIN ONLY"
)

print(
    "  Horizontal flip          : DISABLED"
)

print(
    "  Temporal reversal        : DISABLED"
)

print()

print(
    "RGB CACHE"
)

print(
    f"  Train cached             : {len(train_cache_df)}"
)

print(
    f"    created this run       : {created_train}"
)

print(
    f"    reused                 : {existing_train}"
)

print(
    f"  Validation cached        : {len(val_cache_df)}"
)

print(
    f"    created this run       : {created_val}"
)

print(
    f"    reused                 : {existing_val}"
)

print(
    "  Test cached              : 0"
)

print()

print(
    "TEST LOCK"
)

print(
    "  Test decoded             : NO"
)

print(
    "  Test cached              : NO"
)

print(
    "  Test available to L19    : NO"
)

print()

print(
    "PER-CLASS DATA"
)

print("-" * 110)

print(
    class_summary[
        [
            "deployment_class_id",
            "gloss",
            "train_original",
            "validation_original",
            "test_locked",
            "balanced_draws_per_epoch"
        ]
    ]
    .to_string(
        index=False
    )
)

print("-" * 110)

print()

print(
    "Train manifest:"
)

print(
    TRAIN_OUT
)

print()

print(
    "Validation manifest:"
)

print(
    VAL_OUT
)

print()

print(
    "LOCKED test manifest:"
)

print(
    TEST_LOCKED_OUT
)

print()

print(
    "Cache report:"
)

print(
    CACHE_REPORT_OUT
)

print()

print(
    "Augmentation config:"
)

print(
    AUGMENTATION_CONFIG_OUT
)

print()

print(
    "L19 dataset config:"
)

print(
    L19_CONFIG_OUT
)

print()

print(
    "L18B master freeze:"
)

print(
    MASTER_FREEZE_OUT
)

print()

print(
    "Master SHA256:"
)

print(
    MASTER_SHA
)

print()

print(
    "[PASS] L18B COMPLETE AND FROZEN"
)

print(
    "[PASS] 70 TRAIN + 10 VALIDATION + 10 LOCKED TEST preserved"
)

print(
    "[PASS] Robust sequential MP4 decoding used"
)

print(
    "[PASS] Random MP4 frame seeking disabled"
)

print(
    "[PASS] TRAIN-only augmentation protocol frozen"
)

print(
    "[PASS] Balanced 200-draw training epoch defined"
)

print(
    "[PASS] 70 TRAIN + 10 VALIDATION RGB clips cached"
)

print(
    "[PASS] TEST videos were NOT decoded/cached"
)

print()

print(
    "[LOCK] DO NOT MOVE TEST SAMPLES INTO TRAINING"
)

print(
    "[LOCK] DO NOT USE TEST FOR L19 MODEL SELECTION"
)

print()

print(
    "[NEXT] L19 = 10-CLASS RESTRICTED-VOCABULARY MODEL TRAINING"
)

print("=" * 140)

CISLR — L18B — UPDATED ROBUST VERSION
SAFE 10-CLASS DATASET CONSTRUCTION + ROBUST RGB CACHE

Python: 3.11.15
OpenCV: 5.0.0
Seed: 42
Sequence length: 32
Cache resolution: 256x256
L19 model input: 224x224
Balanced draws/class/epoch: 20

[FIX] Sequential MP4 decoding enabled.
[FIX] CAP_PROP_POS_FRAMES random seeking is NOT used.

1. REQUIRED FROZEN ARTIFACT CHECK
L17 vocabulary           : PASS
L17 freeze               : PASS
L18A freeze              : PASS
L18A all                 : PASS
L18A train               : PASS
L18A validation          : PASS
L18A test                : PASS

[PASS] Required frozen inputs exist.

2. FROZEN DEPLOYMENT VOCABULARY
 0  monday
 1  interest
 2  thursday
 3  christmas
 4  pencil
 5  mother
 6  tuesday
 7  friday
 8  rubber
 9  wash

3. VERIFYING L18A FREEZE
L18A status: COMPLETE_AND_FROZEN
metadata            :  90 PASS
found               :  90 PASS
readable            :  90 PASS
usable              :  90 PASS
l9_train            :  70 PASS
l9_validatio

In [8]:
# ==================================================================================================
# CISLR — L19
# 10-CLASS RESTRICTED-VOCABULARY HIGH-ACCURACY VIDEO SIGN RECOGNITION
#
# MODEL:
#   Pretrained VideoMAE
#   MCG-NJU/videomae-base-finetuned-kinetics
#
# INPUT:
#   L18B RGB caches
#
# TRAIN:
#   70 independent CISLR videos
#   class-balanced sampling
#   TRAIN-only augmentation
#
# VALIDATION:
#   10 independent CISLR videos
#   exactly 1 / class
#   NO augmentation
#
# TEST:
#   COMPLETELY LOCKED
#   NOT LOADED
#   NOT DECODED
#   NOT CACHED
#   NOT USED
#
# TRAINING:
#   Stage 1 : classifier head only
#   Stage 2 : final transformer blocks + head
#   Stage 3 : optional broader fine-tuning
#
# MODEL SELECTION:
#   Validation accuracy
#   tie-break -> validation loss
#
# IMPORTANT:
#   90% validation = 9 / 10 correct
#   100% validation = 10 / 10 correct
#
# ==================================================================================================


# ==================================================================================================
# 0. IMPORTS
# ==================================================================================================

import os
import gc
import json
import math
import time
import random
import hashlib
import platform
import warnings

from pathlib import Path
from datetime import datetime
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    Dataset,
    DataLoader,
    WeightedRandomSampler
)

import torchvision.transforms.functional as TF
from torchvision.transforms import InterpolationMode

from transformers import VideoMAEForVideoClassification

warnings.filterwarnings("ignore")


# ==================================================================================================
# 1. GLOBAL CONFIGURATION
# ==================================================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# --------------------------------------------------------------------------------------------------
# Determinism
# --------------------------------------------------------------------------------------------------

torch.backends.cudnn.deterministic = False
torch.backends.cudnn.benchmark = True


# --------------------------------------------------------------------------------------------------
# Device
# --------------------------------------------------------------------------------------------------

DEVICE = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


# ==================================================================================================
# 2. PATHS
# ==================================================================================================

PROJECT_ROOT = Path(
    "/home/dipshikha/Music/cao/CISLR_RESEARCH"
)

ROOT = (
    PROJECT_ROOT /
    "CISLR_LANDMARK_MODEL"
)


# --------------------------------------------------------------------------------------------------
# L18B
# --------------------------------------------------------------------------------------------------

L18B_ROOT = (
    ROOT /
    "audit/l18b_10class_dataset"
)

L18B_FREEZE = (
    L18B_ROOT /
    "L18B_MASTER_FREEZE.json"
)

TRAIN_MANIFEST = (
    L18B_ROOT /
    "manifests/l18b_train_manifest.csv"
)

VAL_MANIFEST = (
    L18B_ROOT /
    "manifests/l18b_validation_manifest.csv"
)

CLASS_MAPPING_PATH = (
    L18B_ROOT /
    "config/l18b_class_mapping.json"
)

L19_DATASET_CONFIG = (
    L18B_ROOT /
    "config/l19_dataset_config.json"
)


# IMPORTANT:
#
# Notice that there is deliberately NO TEST_MANIFEST variable here.
#
# L19 does not need it.


# --------------------------------------------------------------------------------------------------
# L19 output
# --------------------------------------------------------------------------------------------------

L19_ROOT = (
    ROOT /
    "audit/l19_10class_videomae"
)

MODEL_DIR = (
    L19_ROOT /
    "models"
)

REPORT_DIR = (
    L19_ROOT /
    "reports"
)

PREDICTION_DIR = (
    L19_ROOT /
    "predictions"
)

HASH_DIR = (
    L19_ROOT /
    "hashes"
)


for path in [
    L19_ROOT,
    MODEL_DIR,
    REPORT_DIR,
    PREDICTION_DIR,
    HASH_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True
    )


BEST_MODEL_PATH = (
    MODEL_DIR /
    "l19_best_videomae_10class.pt"
)

LAST_MODEL_PATH = (
    MODEL_DIR /
    "l19_last_videomae_10class.pt"
)

TRAIN_HISTORY_PATH = (
    REPORT_DIR /
    "l19_training_history.csv"
)

VAL_PREDICTIONS_PATH = (
    PREDICTION_DIR /
    "l19_validation_predictions.csv"
)

CONFUSION_MATRIX_PATH = (
    REPORT_DIR /
    "l19_validation_confusion_matrix.csv"
)

SUMMARY_PATH = (
    REPORT_DIR /
    "l19_summary.json"
)

HASH_PATH = (
    HASH_DIR /
    "l19_sha256_manifest.json"
)

FREEZE_PATH = (
    L19_ROOT /
    "L19_MASTER_FREEZE.json"
)


# ==================================================================================================
# 3. MODEL / TRAINING CONFIGURATION
# ==================================================================================================

MODEL_NAME = (
    "MCG-NJU/videomae-base-finetuned-kinetics"
)

NUM_CLASSES = 10


# --------------------------------------------------------------------------------------------------
# VideoMAE expects 16 frames.
#
# L18B caches 32 frames.
#
# During training:
#     select 16 frames from the cached 32.
#
# During validation:
#     deterministic 16-frame sampling.
# --------------------------------------------------------------------------------------------------

CACHE_FRAMES = 32
MODEL_FRAMES = 16

CACHE_SIZE = 256
INPUT_SIZE = 224


# --------------------------------------------------------------------------------------------------
# Batch size
#
# RTX 4050 Laptop GPU ~6 GB:
#
# VideoMAE base is relatively heavy.
#
# Start with batch size 2.
#
# Gradient accumulation gives effective batch 8.
# --------------------------------------------------------------------------------------------------

BATCH_SIZE = 2

GRAD_ACCUM_STEPS = 4

NUM_WORKERS = 2


# --------------------------------------------------------------------------------------------------
# Training stages
# --------------------------------------------------------------------------------------------------

STAGE1_EPOCHS = 15

STAGE2_EPOCHS = 35

STAGE3_EPOCHS = 20

MAX_TOTAL_EPOCHS = (
    STAGE1_EPOCHS +
    STAGE2_EPOCHS +
    STAGE3_EPOCHS
)


# --------------------------------------------------------------------------------------------------
# Learning rates
# --------------------------------------------------------------------------------------------------

LR_STAGE1 = 3e-4

LR_STAGE2 = 2e-5

LR_STAGE3 = 5e-6


WEIGHT_DECAY = 1e-4


# --------------------------------------------------------------------------------------------------
# Early stopping
# --------------------------------------------------------------------------------------------------

EARLY_STOP_PATIENCE = 12


# --------------------------------------------------------------------------------------------------
# Label smoothing
# --------------------------------------------------------------------------------------------------

LABEL_SMOOTHING = 0.05


# --------------------------------------------------------------------------------------------------
# AMP
# --------------------------------------------------------------------------------------------------

USE_AMP = (
    DEVICE.type ==
    "cuda"
)


# ==================================================================================================
# 4. HELPERS
# ==================================================================================================

def sha256_file(path):

    h = hashlib.sha256()

    with open(path, "rb") as f:

        while True:

            block = f.read(
                1024 * 1024
            )

            if not block:
                break

            h.update(block)

    return h.hexdigest()


def save_json(
    obj,
    path
):

    with open(
        path,
        "w"
    ) as f:

        json.dump(
            obj,
            f,
            indent=2,
            ensure_ascii=False
        )


def seed_worker(worker_id):

    worker_seed = (
        SEED +
        worker_id
    )

    np.random.seed(
        worker_seed
    )

    random.seed(
        worker_seed
    )


# ==================================================================================================
# 5. HEADER
# ==================================================================================================

print("=" * 140)

print(
    "CISLR — L19"
)

print(
    "10-CLASS RESTRICTED-VOCABULARY VideoMAE TRAINING"
)

print("=" * 140)

print()

print(
    "Python:",
    platform.python_version()
)

print(
    "PyTorch:",
    torch.__version__
)

print(
    "Device:",
    DEVICE
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

    props = torch.cuda.get_device_properties(
        0
    )

    print(
        "GPU memory:",
        round(
            props.total_memory /
            1024**3,
            2
        ),
        "GB"
    )

print()

print(
    "Model:",
    MODEL_NAME
)

print(
    "Classes:",
    NUM_CLASSES
)

print(
    "Cached frames:",
    CACHE_FRAMES
)

print(
    "Model frames:",
    MODEL_FRAMES
)

print(
    "Input:",
    f"{INPUT_SIZE}x{INPUT_SIZE}"
)

print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Gradient accumulation:",
    GRAD_ACCUM_STEPS
)

print(
    "Effective batch:",
    BATCH_SIZE *
    GRAD_ACCUM_STEPS
)


# ==================================================================================================
# 6. VERIFY L18B
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "1. VERIFYING FROZEN L18B"
)

print(
    "=" * 140
)


required_files = {

    "L18B freeze":
        L18B_FREEZE,

    "train manifest":
        TRAIN_MANIFEST,

    "validation manifest":
        VAL_MANIFEST,

    "class mapping":
        CLASS_MAPPING_PATH,

    "L19 dataset config":
        L19_DATASET_CONFIG,
}


for name, path in (
    required_files.items()
):

    exists = path.exists()

    print(
        f"{name:25s}: "
        f"{'PASS' if exists else 'FAIL'}"
    )

    if not exists:

        raise FileNotFoundError(
            path
        )


with open(
    L18B_FREEZE,
    "r"
) as f:

    l18b_freeze = json.load(
        f
    )


if (
    l18b_freeze.get(
        "status"
    )
    !=
    "COMPLETE_AND_FROZEN"
):

    raise RuntimeError(
        "L18B is not COMPLETE_AND_FROZEN."
    )


population = l18b_freeze[
    "population"
]


if int(
    population[
        "train"
    ]
) != 70:

    raise RuntimeError(
        "L18B train count != 70."
    )


if int(
    population[
        "validation"
    ]
) != 10:

    raise RuntimeError(
        "L18B validation count != 10."
    )


if int(
    population[
        "test"
    ]
) != 10:

    raise RuntimeError(
        "L18B test count != 10."
    )


test_lock = l18b_freeze[
    "test_lock"
]


if not bool(
    test_lock[
        "locked"
    ]
):

    raise RuntimeError(
        "L18B test is not locked."
    )


if bool(
    test_lock[
        "available_for_l19_training"
    ]
):

    raise RuntimeError(
        "Test unexpectedly available for L19 training."
    )


if bool(
    test_lock[
        "available_for_l19_model_selection"
    ]
):

    raise RuntimeError(
        "Test unexpectedly available for L19 model selection."
    )


print()

print(
    "[PASS] L18B COMPLETE_AND_FROZEN"
)

print(
    "[PASS] 70 train"
)

print(
    "[PASS] 10 validation"
)

print(
    "[LOCK] 10 test unavailable to L19"
)


# ==================================================================================================
# 7. LOAD CLASS MAPPING
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "2. CLASS MAPPING"
)

print(
    "=" * 140
)


with open(
    CLASS_MAPPING_PATH,
    "r"
) as f:

    class_mapping = json.load(
        f
    )


id_to_gloss = {

    int(k):
        v

    for k, v in (
        class_mapping[
            "class_id_to_gloss"
        ].items()
    )
}


gloss_to_id = {

    k:
        int(v)

    for k, v in (
        class_mapping[
            "gloss_to_class_id"
        ].items()
    )
}


if set(
    id_to_gloss.keys()
) != set(
    range(
        NUM_CLASSES
    )
):

    raise RuntimeError(
        "Class mapping is not 0..9."
    )


for class_id in range(
    NUM_CLASSES
):

    print(
        f"{class_id:2d}  "
        f"{id_to_gloss[class_id]}"
    )


# ==================================================================================================
# 8. LOAD TRAIN / VALIDATION ONLY
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "3. LOAD TRAIN + VALIDATION"
)

print(
    "=" * 140
)


train_df = pd.read_csv(
    TRAIN_MANIFEST
)


val_df = pd.read_csv(
    VAL_MANIFEST
)


if len(
    train_df
) != 70:

    raise RuntimeError(
        "Train manifest does not contain 70 rows."
    )


if len(
    val_df
) != 10:

    raise RuntimeError(
        "Validation manifest does not contain 10 rows."
    )


train_df[
    "deployment_class_id"
] = pd.to_numeric(
    train_df[
        "deployment_class_id"
    ],
    errors="raise"
).astype(int)


val_df[
    "deployment_class_id"
] = pd.to_numeric(
    val_df[
        "deployment_class_id"
    ],
    errors="raise"
).astype(int)


# --------------------------------------------------------------------------------------------------
# Verify cache paths
# --------------------------------------------------------------------------------------------------

for split_name, df in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df)
]:

    for p in df[
        "rgb_cache_path"
    ]:

        p = Path(
            p
        )

        if not p.exists():

            raise FileNotFoundError(
                f"{split_name} cache missing: {p}"
            )


print(
    "Train rows:",
    len(
        train_df
    )
)

print(
    "Validation rows:",
    len(
        val_df
    )
)


print()

print(
    "Train class counts:"
)

print(
    train_df[
        "deployment_class_id"
    ]
    .value_counts()
    .sort_index()
)


print()

print(
    "[PASS] Only TRAIN and VALIDATION manifests loaded."
)

print(
    "[LOCK] TEST manifest has NOT been loaded."
)


# ==================================================================================================
# 9. TEMPORAL SAMPLING
# ==================================================================================================

def temporal_sample_train(
    clip
):
    """
    clip:
        numpy uint8
        [32, H, W, 3]

    Return:
        16 frames

    Mild temporal jitter while preserving order.
    """

    total = clip.shape[0]


    if total < MODEL_FRAMES:

        indices = np.linspace(
            0,
            total - 1,
            MODEL_FRAMES
        )

        indices = np.round(
            indices
        ).astype(int)

        return clip[
            indices
        ]


    # ------------------------------------------------------------------
    # Divide timeline into 16 bins.
    #
    # Pick one frame randomly from each bin.
    #
    # Preserves temporal order.
    # ------------------------------------------------------------------

    boundaries = np.linspace(
        0,
        total,
        MODEL_FRAMES + 1
    )


    indices = []


    for i in range(
        MODEL_FRAMES
    ):

        start = int(
            math.floor(
                boundaries[i]
            )
        )

        end = int(
            math.floor(
                boundaries[i + 1]
            )
        )


        end = max(
            end,
            start + 1
        )


        end = min(
            end,
            total
        )


        index = random.randrange(
            start,
            end
        )


        indices.append(
            index
        )


    return clip[
        np.array(
            indices
        )
    ]


def temporal_sample_validation(
    clip
):

    total = clip.shape[0]


    indices = np.linspace(
        0,
        total - 1,
        MODEL_FRAMES
    )


    indices = np.round(
        indices
    ).astype(
        np.int64
    )


    return clip[
        indices
    ]


# ==================================================================================================
# 10. CONSISTENT VIDEO AUGMENTATION
#
# The SAME spatial / photometric transform is applied to every frame
# of a clip so temporal consistency is preserved.
# ==================================================================================================

def augment_video_train(
    clip
):
    """
    Input:
        uint8 numpy
        [T, 256, 256, 3]

    Output:
        float tensor
        [T, 3, 224, 224]
    """

    # ----------------------------------------------------------------------------------------------
    # numpy -> torch
    # [T,H,W,C] -> [T,C,H,W]
    # ----------------------------------------------------------------------------------------------

    x = torch.from_numpy(
        clip.copy()
    )

    x = x.permute(
        0,
        3,
        1,
        2
    ).float() / 255.0


    # ----------------------------------------------------------------------------------------------
    # Random crop parameters
    # ----------------------------------------------------------------------------------------------

    scale = random.uniform(
        0.82,
        1.00
    )


    crop_size = int(
        round(
            CACHE_SIZE *
            math.sqrt(
                scale
            )
        )
    )


    crop_size = max(
        INPUT_SIZE,
        min(
            CACHE_SIZE,
            crop_size
        )
    )


    max_offset = (
        CACHE_SIZE -
        crop_size
    )


    if max_offset > 0:

        top = random.randint(
            0,
            max_offset
        )

        left = random.randint(
            0,
            max_offset
        )

    else:

        top = 0
        left = 0


    x = x[
        :,
        :,
        top:
        top + crop_size,
        left:
        left + crop_size
    ]


    x = F.interpolate(
        x,
        size=(
            INPUT_SIZE,
            INPUT_SIZE
        ),
        mode="bilinear",
        align_corners=False
    )


    # ----------------------------------------------------------------------------------------------
    # Mild affine transform
    #
    # SAME parameters for all frames.
    # ----------------------------------------------------------------------------------------------

    angle = random.uniform(
        -5.0,
        5.0
    )


    max_translate = int(
        INPUT_SIZE *
        0.04
    )


    translate_x = random.randint(
        -max_translate,
        max_translate
    )


    translate_y = random.randint(
        -max_translate,
        max_translate
    )


    transformed = []


    for frame in x:

        frame = TF.affine(
            frame,
            angle=angle,
            translate=[
                translate_x,
                translate_y
            ],
            scale=1.0,
            shear=[
                0.0,
                0.0
            ],
            interpolation=
                InterpolationMode.BILINEAR
        )

        transformed.append(
            frame
        )


    x = torch.stack(
        transformed,
        dim=0
    )


    # ----------------------------------------------------------------------------------------------
    # Photometric jitter
    #
    # SAME factors across whole clip.
    # ----------------------------------------------------------------------------------------------

    brightness = random.uniform(
        0.85,
        1.15
    )


    contrast = random.uniform(
        0.85,
        1.15
    )


    saturation = random.uniform(
        0.90,
        1.10
    )


    x = TF.adjust_brightness(
        x,
        brightness
    )


    x = TF.adjust_contrast(
        x,
        contrast
    )


    x = TF.adjust_saturation(
        x,
        saturation
    )


    x = torch.clamp(
        x,
        0.0,
        1.0
    )


    return x


def transform_video_validation(
    clip
):

    x = torch.from_numpy(
        clip.copy()
    )

    x = x.permute(
        0,
        3,
        1,
        2
    ).float() / 255.0


    # ----------------------------------------------------------------------------------------------
    # Deterministic center crop
    # ----------------------------------------------------------------------------------------------

    top = (
        CACHE_SIZE -
        INPUT_SIZE
    ) // 2


    left = (
        CACHE_SIZE -
        INPUT_SIZE
    ) // 2


    x = x[
        :,
        :,
        top:
        top + INPUT_SIZE,
        left:
        left + INPUT_SIZE
    ]


    return x


# ==================================================================================================
# 11. VIDEOMAE NORMALIZATION
#
# ImageNet mean/std used by the pretrained VideoMAE processor.
# ==================================================================================================

IMAGE_MEAN = torch.tensor(
    [
        0.485,
        0.456,
        0.406
    ]
).view(
    1,
    3,
    1,
    1
)


IMAGE_STD = torch.tensor(
    [
        0.229,
        0.224,
        0.225
    ]
).view(
    1,
    3,
    1,
    1
)


def normalize_video(
    x
):

    return (
        x -
        IMAGE_MEAN
    ) / IMAGE_STD


# ==================================================================================================
# 12. DATASET
# ==================================================================================================

class CISLR10Dataset(
    Dataset
):

    def __init__(
        self,
        dataframe,
        train=False
    ):

        self.df = (
            dataframe
            .reset_index(
                drop=True
            )
            .copy()
        )

        self.train = train


    def __len__(
        self
    ):

        return len(
            self.df
        )


    def __getitem__(
        self,
        index
    ):

        row = self.df.iloc[
            index
        ]


        cache_path = Path(
            row[
                "rgb_cache_path"
            ]
        )


        clip = np.load(
            cache_path
        )


        if clip.shape != (
            CACHE_FRAMES,
            CACHE_SIZE,
            CACHE_SIZE,
            3
        ):

            raise RuntimeError(
                f"Unexpected cache shape "
                f"{clip.shape}: {cache_path}"
            )


        # ------------------------------------------------------------------------------------------
        # Temporal
        # ------------------------------------------------------------------------------------------

        if self.train:

            clip = temporal_sample_train(
                clip
            )

        else:

            clip = temporal_sample_validation(
                clip
            )


        # ------------------------------------------------------------------------------------------
        # Spatial / photometric
        # ------------------------------------------------------------------------------------------

        if self.train:

            x = augment_video_train(
                clip
            )

        else:

            x = transform_video_validation(
                clip
            )


        # ------------------------------------------------------------------------------------------
        # Normalize
        # ------------------------------------------------------------------------------------------

        x = normalize_video(
            x
        )


        # ------------------------------------------------------------------------------------------
        # VideoMAE expects:
        #
        # [T, C, H, W]
        #
        # DataLoader -> [B,T,C,H,W]
        # ------------------------------------------------------------------------------------------

        label = int(
            row[
                "deployment_class_id"
            ]
        )


        uid = str(
            row[
                "uid"
            ]
        )


        gloss = str(
            row[
                "gloss"
            ]
        )


        return {

            "pixel_values":
                x,

            "label":
                torch.tensor(
                    label,
                    dtype=torch.long
                ),

            "uid":
                uid,

            "gloss":
                gloss
        }


# ==================================================================================================
# 13. DATASETS
# ==================================================================================================

train_dataset = CISLR10Dataset(
    train_df,
    train=True
)


val_dataset = CISLR10Dataset(
    val_df,
    train=False
)


# ==================================================================================================
# 14. BALANCED SAMPLER
#
# Each class receives equal expected probability.
#
# Number of draws / epoch = 200.
# ==================================================================================================

class_counts = (
    train_df[
        "deployment_class_id"
    ]
    .value_counts()
    .to_dict()
)


sample_weights = []


for label in train_df[
    "deployment_class_id"
]:

    sample_weights.append(
        1.0 /
        class_counts[
            int(
                label
            )
        ]
    )


sample_weights = torch.tensor(
    sample_weights,
    dtype=torch.double
)


sampler_generator = torch.Generator()

sampler_generator.manual_seed(
    SEED
)


train_sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=200,
    replacement=True,
    generator=sampler_generator
)


# ==================================================================================================
# 15. DATALOADERS
# ==================================================================================================

worker_generator = torch.Generator()

worker_generator.manual_seed(
    SEED
)


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    sampler=train_sampler,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type ==
        "cuda"
    ),
    worker_init_fn=seed_worker,
    generator=worker_generator,
    persistent_workers=(
        NUM_WORKERS > 0
    )
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(
        DEVICE.type ==
        "cuda"
    ),
    worker_init_fn=seed_worker,
    persistent_workers=(
        NUM_WORKERS > 0
    )
)


print(
    "\nTrain dataset:",
    len(
        train_dataset
    )
)

print(
    "Train draws/epoch:",
    len(
        train_sampler
    )
)

print(
    "Validation dataset:",
    len(
        val_dataset
    )
)


# ==================================================================================================
# 16. DATA SANITY CHECK
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "4. DATA SANITY CHECK"
)

print(
    "=" * 140
)


sample_batch = next(
    iter(
        train_loader
    )
)


print(
    "pixel_values:",
    sample_batch[
        "pixel_values"
    ].shape
)

print(
    "labels:",
    sample_batch[
        "label"
    ]
)

print(
    "uids:",
    sample_batch[
        "uid"
    ]
)


expected_batch_shape = (
    BATCH_SIZE,
    MODEL_FRAMES,
    3,
    INPUT_SIZE,
    INPUT_SIZE
)


if (
    tuple(
        sample_batch[
            "pixel_values"
        ].shape
    )
    !=
    expected_batch_shape
):

    raise RuntimeError(
        "Unexpected model input shape."
    )


print()

print(
    "[PASS] Input tensor shape correct."
)


del sample_batch

gc.collect()


# ==================================================================================================
# 17. LOAD PRETRAINED VIDEOMAE
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "5. LOADING PRETRAINED VIDEOMAE"
)

print(
    "=" * 140
)


model = (
    VideoMAEForVideoClassification
    .from_pretrained(
        MODEL_NAME,
        num_labels=NUM_CLASSES,
        ignore_mismatched_sizes=True,
        label2id={
            id_to_gloss[i]:
                i
            for i in range(
                NUM_CLASSES
            )
        },
        id2label={
            i:
                id_to_gloss[i]
            for i in range(
                NUM_CLASSES
            )
        }
    )
)


model = model.to(
    DEVICE
)


total_params = sum(
    p.numel()
    for p in model.parameters()
)


trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)


print(
    "Total parameters:",
    f"{total_params:,}"
)

print(
    "Initial trainable:",
    f"{trainable_params:,}"
)


# ==================================================================================================
# 18. LOSS
# ==================================================================================================

criterion = nn.CrossEntropyLoss(
    label_smoothing=
        LABEL_SMOOTHING
)


# ==================================================================================================
# 19. AMP SCALER
# ==================================================================================================

try:

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=USE_AMP
    )

except Exception:

    scaler = torch.cuda.amp.GradScaler(
        enabled=USE_AMP
    )


# ==================================================================================================
# 20. PARAMETER FREEZING HELPERS
# ==================================================================================================

def freeze_all(
    model
):

    for p in model.parameters():

        p.requires_grad = False


def configure_stage1(
    model
):
    """
    Classifier head only.
    """

    freeze_all(
        model
    )


    for p in model.classifier.parameters():

        p.requires_grad = True


def configure_stage2(
    model
):
    """
    Final 4 transformer blocks + final layernorm + classifier.
    """

    freeze_all(
        model
    )


    # classifier

    for p in model.classifier.parameters():

        p.requires_grad = True


    # final layer norm

    if hasattr(
        model.videomae.encoder,
        "layernorm"
    ):

        for p in (
            model.videomae.encoder
            .layernorm
            .parameters()
        ):

            p.requires_grad = True


    # transformer blocks

    layers = (
        model.videomae.encoder.layer
    )


    for layer in layers[-4:]:

        for p in layer.parameters():

            p.requires_grad = True


def configure_stage3(
    model
):
    """
    Final 8 transformer blocks + classifier.

    Still avoids fully unfreezing the entire backbone.
    """

    freeze_all(
        model
    )


    for p in model.classifier.parameters():

        p.requires_grad = True


    if hasattr(
        model.videomae.encoder,
        "layernorm"
    ):

        for p in (
            model.videomae.encoder
            .layernorm
            .parameters()
        ):

            p.requires_grad = True


    layers = (
        model.videomae.encoder.layer
    )


    for layer in layers[-8:]:

        for p in layer.parameters():

            p.requires_grad = True


def count_trainable(
    model
):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


# ==================================================================================================
# 21. OPTIMIZER
# ==================================================================================================

def make_optimizer(
    model,
    learning_rate
):

    params = [
        p
        for p in model.parameters()
        if p.requires_grad
    ]


    return torch.optim.AdamW(
        params,
        lr=learning_rate,
        weight_decay=
            WEIGHT_DECAY
    )


# ==================================================================================================
# 22. TRAIN ONE EPOCH
# ==================================================================================================

def train_one_epoch(
    model,
    loader,
    optimizer
):

    model.train()


    running_loss = 0.0

    correct = 0

    total = 0


    optimizer.zero_grad(
        set_to_none=True
    )


    for batch_index, batch in enumerate(
        loader
    ):

        pixel_values = (
            batch[
                "pixel_values"
            ]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        labels = (
            batch[
                "label"
            ]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        # ------------------------------------------------------------------------------------------
        # Forward
        # ------------------------------------------------------------------------------------------

        with torch.autocast(
            device_type=
                DEVICE.type,
            dtype=
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16,
            enabled=USE_AMP
        ):

            outputs = model(
                pixel_values=
                    pixel_values
            )


            logits = outputs.logits


            loss = criterion(
                logits,
                labels
            )


            loss_for_backward = (
                loss /
                GRAD_ACCUM_STEPS
            )


        # ------------------------------------------------------------------------------------------
        # Backward
        # ------------------------------------------------------------------------------------------

        scaler.scale(
            loss_for_backward
        ).backward()


        should_step = (
            (
                batch_index + 1
            )
            %
            GRAD_ACCUM_STEPS
            ==
            0
            or
            (
                batch_index + 1
            )
            ==
            len(
                loader
            )
        )


        if should_step:

            scaler.unscale_(
                optimizer
            )


            torch.nn.utils.clip_grad_norm_(
                [
                    p
                    for p in model.parameters()
                    if p.requires_grad
                ],
                max_norm=1.0
            )


            scaler.step(
                optimizer
            )


            scaler.update()


            optimizer.zero_grad(
                set_to_none=True
            )


        # ------------------------------------------------------------------------------------------
        # Metrics
        # ------------------------------------------------------------------------------------------

        batch_size = labels.size(
            0
        )


        running_loss += (
            float(
                loss.item()
            )
            *
            batch_size
        )


        predictions = torch.argmax(
            logits,
            dim=1
        )


        correct += int(
            (
                predictions
                ==
                labels
            )
            .sum()
            .item()
        )


        total += batch_size


        del (
            pixel_values,
            labels,
            outputs,
            logits,
            loss
        )


    epoch_loss = (
        running_loss /
        max(
            total,
            1
        )
    )


    epoch_accuracy = (
        100.0 *
        correct /
        max(
            total,
            1
        )
    )


    return (
        epoch_loss,
        epoch_accuracy,
        correct,
        total
    )


# ==================================================================================================
# 23. VALIDATION
# ==================================================================================================

@torch.no_grad()
def evaluate(
    model,
    loader
):

    model.eval()


    running_loss = 0.0

    correct = 0

    total = 0


    all_labels = []

    all_predictions = []

    all_probabilities = []

    all_uids = []

    all_glosses = []


    for batch in loader:

        pixel_values = (
            batch[
                "pixel_values"
            ]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        labels = (
            batch[
                "label"
            ]
            .to(
                DEVICE,
                non_blocking=True
            )
        )


        with torch.autocast(
            device_type=
                DEVICE.type,
            dtype=
                torch.float16
                if DEVICE.type == "cuda"
                else torch.bfloat16,
            enabled=USE_AMP
        ):

            outputs = model(
                pixel_values=
                    pixel_values
            )


            logits = outputs.logits


            loss = criterion(
                logits,
                labels
            )


        probabilities = torch.softmax(
            logits.float(),
            dim=1
        )


        predictions = torch.argmax(
            probabilities,
            dim=1
        )


        batch_size = labels.size(
            0
        )


        running_loss += (
            float(
                loss.item()
            )
            *
            batch_size
        )


        correct += int(
            (
                predictions
                ==
                labels
            )
            .sum()
            .item()
        )


        total += batch_size


        all_labels.extend(
            labels.cpu().tolist()
        )


        all_predictions.extend(
            predictions.cpu().tolist()
        )


        all_probabilities.append(
            probabilities.cpu().numpy()
        )


        all_uids.extend(
            list(
                batch[
                    "uid"
                ]
            )
        )


        all_glosses.extend(
            list(
                batch[
                    "gloss"
                ]
            )
        )


    probabilities = np.concatenate(
        all_probabilities,
        axis=0
    )


    epoch_loss = (
        running_loss /
        max(
            total,
            1
        )
    )


    epoch_accuracy = (
        100.0 *
        correct /
        max(
            total,
            1
        )
    )


    return {

        "loss":
            epoch_loss,

        "accuracy":
            epoch_accuracy,

        "correct":
            correct,

        "total":
            total,

        "labels":
            np.array(
                all_labels,
                dtype=np.int64
            ),

        "predictions":
            np.array(
                all_predictions,
                dtype=np.int64
            ),

        "probabilities":
            probabilities,

        "uids":
            all_uids,

        "glosses":
            all_glosses
    }


# ==================================================================================================
# 24. CHECKPOINT SAVER
# ==================================================================================================

def save_checkpoint(
    path,
    model,
    stage,
    epoch,
    global_epoch,
    train_loss,
    train_accuracy,
    val_result
):

    checkpoint = {

        "stage":
            "L19",

        "model_name":
            MODEL_NAME,

        "num_classes":
            NUM_CLASSES,

        "class_mapping":
            id_to_gloss,

        "training_stage":
            stage,

        "epoch":
            epoch,

        "global_epoch":
            global_epoch,

        "train_loss":
            float(
                train_loss
            ),

        "train_accuracy":
            float(
                train_accuracy
            ),

        "validation_loss":
            float(
                val_result[
                    "loss"
                ]
            ),

        "validation_accuracy":
            float(
                val_result[
                    "accuracy"
                ]
            ),

        "validation_correct":
            int(
                val_result[
                    "correct"
                ]
            ),

        "validation_total":
            int(
                val_result[
                    "total"
                ]
            ),

        "model_state_dict":
            model.state_dict(),

        "source_l18b_freeze_sha256":
            sha256_file(
                L18B_FREEZE
            ),

        "created_at":
            datetime.now().isoformat()
    }


    torch.save(
        checkpoint,
        path
    )


# ==================================================================================================
# 25. TRAINING STATE
# ==================================================================================================

history = []


best_val_accuracy = -1.0

best_val_loss = float(
    "inf"
)

best_global_epoch = -1

best_stage = None

best_stage_epoch = None

global_epoch = 0

epochs_without_improvement = 0


# ==================================================================================================
# 26. TRAINING STAGE FUNCTION
# ==================================================================================================

def run_training_stage(
    stage_name,
    epochs,
    learning_rate,
    configure_function
):

    global global_epoch
    global best_val_accuracy
    global best_val_loss
    global best_global_epoch
    global best_stage
    global best_stage_epoch
    global epochs_without_improvement
    global history


    # ----------------------------------------------------------------------------------------------
    # Configure trainable parameters
    # ----------------------------------------------------------------------------------------------

    configure_function(
        model
    )


    trainable = count_trainable(
        model
    )


    optimizer = make_optimizer(
        model,
        learning_rate
    )


    # ----------------------------------------------------------------------------------------------
    # Cosine scheduler
    # ----------------------------------------------------------------------------------------------

    scheduler = (
        torch.optim.lr_scheduler
        .CosineAnnealingLR(
            optimizer,
            T_max=max(
                epochs,
                1
            ),
            eta_min=
                learning_rate *
                0.05
        )
    )


    print(
        "\n" +
        "=" * 140
    )

    print(
        f"{stage_name}"
    )

    print(
        "=" * 140
    )

    print(
        "Epochs:",
        epochs
    )

    print(
        "Learning rate:",
        learning_rate
    )

    print(
        "Trainable parameters:",
        f"{trainable:,}"
    )

    print()


    for stage_epoch in range(
        1,
        epochs + 1
    ):

        global_epoch += 1


        epoch_start = time.time()


        # ------------------------------------------------------------------------------------------
        # Train
        # ------------------------------------------------------------------------------------------

        (
            train_loss,
            train_accuracy,
            train_correct,
            train_total
        ) = train_one_epoch(
            model,
            train_loader,
            optimizer
        )


        # ------------------------------------------------------------------------------------------
        # Validate
        # ------------------------------------------------------------------------------------------

        val_result = evaluate(
            model,
            val_loader
        )


        current_lr = optimizer.param_groups[
            0
        ][
            "lr"
        ]


        scheduler.step()


        elapsed = (
            time.time() -
            epoch_start
        )


        # ------------------------------------------------------------------------------------------
        # Determine improvement
        #
        # Primary:
        #     validation accuracy
        #
        # Tie-break:
        #     lower validation loss
        # ------------------------------------------------------------------------------------------

        improved = False


        if (
            val_result[
                "accuracy"
            ]
            >
            best_val_accuracy
        ):

            improved = True


        elif (
            val_result[
                "accuracy"
            ]
            ==
            best_val_accuracy
            and
            val_result[
                "loss"
            ]
            <
            best_val_loss
        ):

            improved = True


        # ------------------------------------------------------------------------------------------
        # Save best
        # ------------------------------------------------------------------------------------------

        if improved:

            best_val_accuracy = float(
                val_result[
                    "accuracy"
                ]
            )


            best_val_loss = float(
                val_result[
                    "loss"
                ]
            )


            best_global_epoch = (
                global_epoch
            )


            best_stage = (
                stage_name
            )


            best_stage_epoch = (
                stage_epoch
            )


            epochs_without_improvement = 0


            save_checkpoint(
                BEST_MODEL_PATH,
                model,
                stage_name,
                stage_epoch,
                global_epoch,
                train_loss,
                train_accuracy,
                val_result
            )


            marker = "★ BEST"


        else:

            epochs_without_improvement += 1

            marker = ""


        # ------------------------------------------------------------------------------------------
        # History
        # ------------------------------------------------------------------------------------------

        history.append(
            {

                "global_epoch":
                    global_epoch,

                "stage":
                    stage_name,

                "stage_epoch":
                    stage_epoch,

                "learning_rate":
                    current_lr,

                "train_loss":
                    train_loss,

                "train_accuracy":
                    train_accuracy,

                "train_correct":
                    train_correct,

                "train_total":
                    train_total,

                "validation_loss":
                    val_result[
                        "loss"
                    ],

                "validation_accuracy":
                    val_result[
                        "accuracy"
                    ],

                "validation_correct":
                    val_result[
                        "correct"
                    ],

                "validation_total":
                    val_result[
                        "total"
                    ],

                "epoch_seconds":
                    elapsed,

                "best_so_far":
                    improved
            }
        )


        pd.DataFrame(
            history
        ).to_csv(
            TRAIN_HISTORY_PATH,
            index=False
        )


        # ------------------------------------------------------------------------------------------
        # Console
        # ------------------------------------------------------------------------------------------

        print(
            f"Epoch "
            f"{global_epoch:03d} "
            f"| {stage_name:<12s} "
            f"| "
            f"Train "
            f"{train_accuracy:6.2f}% "
            f"({train_correct:3d}/{train_total:3d}) "
            f"| "
            f"Val "
            f"{val_result['accuracy']:6.2f}% "
            f"({val_result['correct']:2d}/{val_result['total']:2d}) "
            f"| "
            f"TLoss {train_loss:.4f} "
            f"| "
            f"VLoss {val_result['loss']:.4f} "
            f"| "
            f"LR {current_lr:.2e} "
            f"| "
            f"{elapsed:6.1f}s "
            f"{marker}"
        )


        # ------------------------------------------------------------------------------------------
        # Clear memory
        # ------------------------------------------------------------------------------------------

        gc.collect()


        if torch.cuda.is_available():

            torch.cuda.empty_cache()


        # ------------------------------------------------------------------------------------------
        # Stage early stop
        # ------------------------------------------------------------------------------------------

        if (
            epochs_without_improvement
            >=
            EARLY_STOP_PATIENCE
        ):

            print()

            print(
                f"[EARLY STOP] "
                f"No validation improvement for "
                f"{EARLY_STOP_PATIENCE} epochs."
            )

            return True


    return False


# ==================================================================================================
# 27. STAGE 1 — CLASSIFIER HEAD
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "6. TRAINING"
)

print(
    "=" * 140
)


stop_training = run_training_stage(
    stage_name=
        "HEAD",

    epochs=
        STAGE1_EPOCHS,

    learning_rate=
        LR_STAGE1,

    configure_function=
        configure_stage1
)


# ==================================================================================================
# 28. STAGE 2 — FINAL 4 BLOCKS
#
# We continue even if Stage 1 plateaued because unfreezing the pretrained
# representation is an intentional predeclared stage transition.
# ==================================================================================================

epochs_without_improvement = 0


stop_training = run_training_stage(
    stage_name=
        "FINETUNE4",

    epochs=
        STAGE2_EPOCHS,

    learning_rate=
        LR_STAGE2,

    configure_function=
        configure_stage2
)


# ==================================================================================================
# 29. STAGE 3 — FINAL 8 BLOCKS
#
# Run only if validation is still below 90%.
#
# IMPORTANT:
# This decision is based only on VALIDATION.
# TEST remains untouched.
# ==================================================================================================

if best_val_accuracy < 90.0:

    print()

    print(
        "[INFO] Best validation accuracy is below 90%."
    )

    print(
        "[INFO] Running predeclared Stage 3 broader fine-tuning."
    )


    epochs_without_improvement = 0


    stop_training = run_training_stage(
        stage_name=
            "FINETUNE8",

        epochs=
            STAGE3_EPOCHS,

        learning_rate=
            LR_STAGE3,

        configure_function=
            configure_stage3
    )


else:

    print()

    print(
        "[INFO] Validation target reached before Stage 3."
    )

    print(
        "[INFO] Stage 3 skipped to avoid unnecessary fine-tuning."
    )


# ==================================================================================================
# 30. SAVE LAST MODEL
# ==================================================================================================

last_val_result = evaluate(
    model,
    val_loader
)


save_checkpoint(
    LAST_MODEL_PATH,
    model,
    "LAST",
    0,
    global_epoch,
    history[-1][
        "train_loss"
    ],
    history[-1][
        "train_accuracy"
    ],
    last_val_result
)


# ==================================================================================================
# 31. LOAD BEST CHECKPOINT
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "7. LOADING BEST VALIDATION CHECKPOINT"
)

print(
    "=" * 140
)


if not BEST_MODEL_PATH.exists():

    raise RuntimeError(
        "Best checkpoint was not created."
    )


best_checkpoint = torch.load(
    BEST_MODEL_PATH,
    map_location=DEVICE,
    weights_only=False
)


model.load_state_dict(
    best_checkpoint[
        "model_state_dict"
    ]
)


model = model.to(
    DEVICE
)


print(
    "Best stage:",
    best_checkpoint[
        "training_stage"
    ]
)

print(
    "Best global epoch:",
    best_checkpoint[
        "global_epoch"
    ]
)

print(
    "Stored validation accuracy:",
    best_checkpoint[
        "validation_accuracy"
    ]
)


# ==================================================================================================
# 32. FINAL VALIDATION EVALUATION
# ==================================================================================================

final_val = evaluate(
    model,
    val_loader
)


print()

print(
    "FINAL BEST VALIDATION"
)

print(
    f"Accuracy : "
    f"{final_val['accuracy']:.6f}%"
)

print(
    f"Correct  : "
    f"{final_val['correct']}/"
    f"{final_val['total']}"
)

print(
    f"Loss     : "
    f"{final_val['loss']:.6f}"
)


# ==================================================================================================
# 33. VALIDATION PREDICTION TABLE
# ==================================================================================================

prediction_rows = []


for i in range(
    len(
        final_val[
            "labels"
        ]
    )
):

    true_id = int(
        final_val[
            "labels"
        ][i]
    )


    pred_id = int(
        final_val[
            "predictions"
        ][i]
    )


    probabilities = (
        final_val[
            "probabilities"
        ][i]
    )


    order = np.argsort(
        probabilities
    )[::-1]


    top1_confidence = float(
        probabilities[
            pred_id
        ]
    )


    top3_ids = order[
        :3
    ]


    prediction_rows.append(
        {

            "uid":
                final_val[
                    "uids"
                ][i],

            "true_class_id":
                true_id,

            "true_gloss":
                id_to_gloss[
                    true_id
                ],

            "predicted_class_id":
                pred_id,

            "predicted_gloss":
                id_to_gloss[
                    pred_id
                ],

            "correct":
                bool(
                    true_id ==
                    pred_id
                ),

            "confidence":
                top1_confidence,

            "top2_gloss":
                id_to_gloss[
                    int(
                        top3_ids[1]
                    )
                ],

            "top2_probability":
                float(
                    probabilities[
                        top3_ids[1]
                    ]
                ),

            "top3_gloss":
                id_to_gloss[
                    int(
                        top3_ids[2]
                    )
                ],

            "top3_probability":
                float(
                    probabilities[
                        top3_ids[2]
                    ]
                )
        }
    )


prediction_df = pd.DataFrame(
    prediction_rows
)


prediction_df.to_csv(
    VAL_PREDICTIONS_PATH,
    index=False
)


print(
    "\nVALIDATION PREDICTIONS"
)

print("-" * 140)

print(
    prediction_df[
        [
            "uid",
            "true_gloss",
            "predicted_gloss",
            "correct",
            "confidence"
        ]
    ].to_string(
        index=False
    )
)

print("-" * 140)


# ==================================================================================================
# 34. CONFUSION MATRIX
# ==================================================================================================

confusion = np.zeros(
    (
        NUM_CLASSES,
        NUM_CLASSES
    ),
    dtype=np.int64
)


for true_id, pred_id in zip(
    final_val[
        "labels"
    ],
    final_val[
        "predictions"
    ]
):

    confusion[
        int(
            true_id
        ),
        int(
            pred_id
        )
    ] += 1


confusion_df = pd.DataFrame(
    confusion,
    index=[
        id_to_gloss[i]
        for i in range(
            NUM_CLASSES
        )
    ],
    columns=[
        id_to_gloss[i]
        for i in range(
            NUM_CLASSES
        )
    ]
)


confusion_df.to_csv(
    CONFUSION_MATRIX_PATH
)


# ==================================================================================================
# 35. TRAINING HISTORY SUMMARY
# ==================================================================================================

history_df = pd.DataFrame(
    history
)


history_df.to_csv(
    TRAIN_HISTORY_PATH,
    index=False
)


best_history_row = history_df.iloc[
    history_df[
        "validation_accuracy"
    ].idxmax()
]


max_train_accuracy = float(
    history_df[
        "train_accuracy"
    ].max()
)


print(
    "\nTRAINING SUMMARY"
)

print(
    "Total epochs executed:",
    len(
        history_df
    )
)

print(
    f"Maximum training accuracy: "
    f"{max_train_accuracy:.2f}%"
)

print(
    f"Best validation accuracy : "
    f"{final_val['accuracy']:.2f}% "
    f"({final_val['correct']}/10)"
)


# ==================================================================================================
# 36. TARGET INTERPRETATION
# ==================================================================================================

if final_val[
    "correct"
] >= 9:

    target_status = (
        "TARGET_REACHED"
    )

else:

    target_status = (
        "TARGET_NOT_REACHED"
    )


print()

if final_val[
    "correct"
] == 10:

    print(
        "[RESULT] 10/10 validation predictions correct = 100%."
    )

elif final_val[
    "correct"
] == 9:

    print(
        "[RESULT] 9/10 validation predictions correct = 90%."
    )

else:

    print(
        f"[RESULT] "
        f"{final_val['correct']}/10 validation predictions correct "
        f"= {final_val['accuracy']:.2f}%."
    )


print(
    "[NOTE] Validation n=10; accuracy resolution is 10 percentage points."
)


# ==================================================================================================
# 37. SUMMARY JSON
# ==================================================================================================

summary = {

    "stage":
        "L19",

    "status":
        "COMPLETE",

    "task":
        (
            "10-class restricted-vocabulary "
            "CISLR sign recognition"
        ),

    "model":
        MODEL_NAME,

    "num_classes":
        NUM_CLASSES,

    "source_data": {

        "train_independent_videos":
            70,

        "validation_videos":
            10,

        "test_videos":
            10,

        "test_used":
            False
    },

    "training": {

        "epochs_executed":
            len(
                history_df
            ),

        "maximum_training_accuracy_percent":
            max_train_accuracy,

        "balanced_draws_per_epoch":
            200,

        "augmentation":
            "TRAIN_ONLY"
    },

    "best_model": {

        "stage":
            best_checkpoint[
                "training_stage"
            ],

        "global_epoch":
            int(
                best_checkpoint[
                    "global_epoch"
                ]
            ),

        "validation_accuracy_percent":
            float(
                final_val[
                    "accuracy"
                ]
            ),

        "validation_correct":
            int(
                final_val[
                    "correct"
                ]
            ),

        "validation_total":
            int(
                final_val[
                    "total"
                ]
            ),

        "validation_loss":
            float(
                final_val[
                    "loss"
                ]
            )
    },

    "target": {

        "requested_threshold_percent":
            90.0,

        "status":
            target_status,

        "scientific_interpretation":
            (
                "Validation contains only 10 videos, "
                "one per deployment class. "
                "A 90% result corresponds to 9/10 correct."
            )
    },

    "test_lock": {

        "test_manifest_loaded":
            False,

        "test_videos_decoded":
            False,

        "test_predictions_generated":
            False,

        "test_used_for_training":
            False,

        "test_used_for_model_selection":
            False
    },

    "next_stage":
        (
            "L20 one-time deployment test "
            "only after L19 model/protocol freeze"
        ),

    "created_at":
        datetime.now().isoformat()
}


save_json(
    summary,
    SUMMARY_PATH
)


# ==================================================================================================
# 38. HASH ARTIFACTS
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "8. ARTIFACT HASHING"
)

print(
    "=" * 140
)


artifact_paths = {

    "best_checkpoint":
        BEST_MODEL_PATH,

    "last_checkpoint":
        LAST_MODEL_PATH,

    "training_history":
        TRAIN_HISTORY_PATH,

    "validation_predictions":
        VAL_PREDICTIONS_PATH,

    "validation_confusion_matrix":
        CONFUSION_MATRIX_PATH,

    "summary":
        SUMMARY_PATH,
}


hash_manifest = {}


for name, path in (
    artifact_paths.items()
):

    digest = sha256_file(
        path
    )


    hash_manifest[
        name
    ] = {

        "path":
            str(
                path
            ),

        "sha256":
            digest
    }


    print(
        f"{name:30s}: "
        f"{digest}"
    )


save_json(
    hash_manifest,
    HASH_PATH
)


# ==================================================================================================
# 39. MASTER FREEZE
# ==================================================================================================

master_freeze = {

    "stage":
        "L19",

    "status":
        "COMPLETE_AND_FROZEN",

    "model":
        MODEL_NAME,

    "num_classes":
        NUM_CLASSES,

    "source_l18b_freeze_sha256":
        sha256_file(
            L18B_FREEZE
        ),

    "best_checkpoint": {

        "path":
            str(
                BEST_MODEL_PATH
            ),

        "sha256":
            sha256_file(
                BEST_MODEL_PATH
            ),

        "training_stage":
            best_checkpoint[
                "training_stage"
            ],

        "global_epoch":
            int(
                best_checkpoint[
                    "global_epoch"
                ]
            )
    },

    "validation": {

        "accuracy_percent":
            float(
                final_val[
                    "accuracy"
                ]
            ),

        "correct":
            int(
                final_val[
                    "correct"
                ]
            ),

        "total":
            10,

        "loss":
            float(
                final_val[
                    "loss"
                ]
            )
    },

    "test": {

        "locked":
            True,

        "loaded":
            False,

        "decoded":
            False,

        "evaluated":
            False,

        "used_for_model_selection":
            False
    },

    "next_stage":
        "L20_ONE_TIME_DEPLOYMENT_TEST",

    "created_at":
        datetime.now().isoformat()
}


save_json(
    master_freeze,
    FREEZE_PATH
)


MASTER_SHA = sha256_file(
    FREEZE_PATH
)


# ==================================================================================================
# 40. FINAL INTEGRITY
# ==================================================================================================

print(
    "\n" +
    "=" * 140
)

print(
    "9. FINAL L19 INTEGRITY CHECK"
)

print(
    "=" * 140
)


checks = {

    "best_checkpoint_exists":
        BEST_MODEL_PATH.exists(),

    "training_history_exists":
        TRAIN_HISTORY_PATH.exists(),

    "validation_predictions_exist":
        VAL_PREDICTIONS_PATH.exists(),

    "validation_confusion_exists":
        CONFUSION_MATRIX_PATH.exists(),

    "summary_exists":
        SUMMARY_PATH.exists(),

    "freeze_exists":
        FREEZE_PATH.exists(),

    "validation_n_10":
        final_val[
            "total"
        ] == 10,

    "test_manifest_not_loaded":
        True,

    "test_not_evaluated":
        True,

    "test_not_used_for_selection":
        True
}


for name, passed in (
    checks.items()
):

    print(
        f"{name:40s}: "
        f"{'PASS' if passed else 'FAIL'}"
    )


failed = [
    name
    for name, passed
    in checks.items()
    if not passed
]


if failed:

    raise RuntimeError(
        f"L19 integrity failure: {failed}"
    )


# ==================================================================================================
# 41. FINAL REPORT
# ==================================================================================================

print("\n")

print("=" * 140)

print(
    "L19 — 10-CLASS RESTRICTED-VOCABULARY TRAINING COMPLETE"
)

print("=" * 140)

print()

print(
    "MODEL"
)

print(
    f"  Backbone                 : {MODEL_NAME}"
)

print(
    "  Classes                  : 10"
)

print(
    f"  Best stage               : {best_checkpoint['training_stage']}"
)

print(
    f"  Best global epoch        : {best_checkpoint['global_epoch']}"
)

print()

print(
    "DATA"
)

print(
    "  Independent train videos : 70"
)

print(
    "  Validation videos        : 10"
)

print(
    "  Locked test videos       : 10"
)

print(
    "  Balanced train draws     : 200 / epoch"
)

print()

print(
    "RESULTS"
)

print(
    f"  Max training accuracy    : {max_train_accuracy:.2f}%"
)

print(
    f"  Validation accuracy      : {final_val['accuracy']:.2f}%"
)

print(
    f"  Validation correct       : {final_val['correct']}/10"
)

print(
    f"  Validation loss          : {final_val['loss']:.6f}"
)

print()

print(
    "TARGET"
)

print(
    "  Requested validation     : >= 90%"
)

print(
    f"  Status                   : {target_status}"
)

print()

print(
    "TEST INTEGRITY"
)

print(
    "  Test manifest loaded     : NO"
)

print(
    "  Test video decoded       : NO"
)

print(
    "  Test inference           : NO"
)

print(
    "  Test used for selection  : NO"
)

print()

print(
    "ARTIFACTS"
)

print(
    "  Best checkpoint:"
)

print(
    f"    {BEST_MODEL_PATH}"
)

print()

print(
    "  Training history:"
)

print(
    f"    {TRAIN_HISTORY_PATH}"
)

print()

print(
    "  Validation predictions:"
)

print(
    f"    {VAL_PREDICTIONS_PATH}"
)

print()

print(
    "  Validation confusion matrix:"
)

print(
    f"    {CONFUSION_MATRIX_PATH}"
)

print()

print(
    "  L19 freeze:"
)

print(
    f"    {FREEZE_PATH}"
)

print()

print(
    "  L19 MASTER SHA256:"
)

print(
    f"    {MASTER_SHA}"
)

print()

print(
    "[PASS] L19 TRAINING COMPLETE"
)

print(
    "[PASS] BEST CHECKPOINT SELECTED USING VALIDATION ONLY"
)

print(
    "[PASS] TEST PARTITION REMAINED LOCKED"
)

print()

if final_val[
    "correct"
] >= 9:

    print(
        "[PASS] >=90% RESTRICTED-VOCABULARY VALIDATION TARGET REACHED"
    )

else:

    print(
        "[INFO] >=90% VALIDATION TARGET NOT YET REACHED"
    )

print()

print(
    "[LOCK] DO NOT RETRAIN USING THE 10 TEST VIDEOS"
)

print(
    "[NEXT] Inspect L19 validation result before running L20."
)

print("=" * 140)

CISLR — L19
10-CLASS RESTRICTED-VOCABULARY VideoMAE TRAINING

Python: 3.11.15
PyTorch: 2.14.0+cu130
Device: cuda
GPU: NVIDIA GeForce RTX 4050 Laptop GPU
GPU memory: 5.64 GB

Model: MCG-NJU/videomae-base-finetuned-kinetics
Classes: 10
Cached frames: 32
Model frames: 16
Input: 224x224
Batch size: 2
Gradient accumulation: 4
Effective batch: 8

1. VERIFYING FROZEN L18B
L18B freeze              : PASS
train manifest           : PASS
validation manifest      : PASS
class mapping            : PASS
L19 dataset config       : PASS

[PASS] L18B COMPLETE_AND_FROZEN
[PASS] 70 train
[PASS] 10 validation
[LOCK] 10 test unavailable to L19

2. CLASS MAPPING
 0  monday
 1  interest
 2  thursday
 3  christmas
 4  pencil
 5  mother
 6  tuesday
 7  friday
 8  rubber
 9  wash

3. LOAD TRAIN + VALIDATION
Train rows: 70
Validation rows: 10

Train class counts:
deployment_class_id
0    11
1     8
2     9
3     6
4     7
5     6
6     6
7     7
8     5
9     5
Name: count, dtype: int64

[PASS] Only TRAIN and V

Some weights of VideoMAEForVideoClassification were not initialized from the model checkpoint at MCG-NJU/videomae-base-finetuned-kinetics and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([400]) in the checkpoint and torch.Size([10]) in the model instantiated
- classifier.weight: found shape torch.Size([400, 768]) in the checkpoint and torch.Size([10, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Total parameters: 86,234,890
Initial trainable: 86,234,890

6. TRAINING

HEAD
Epochs: 15
Learning rate: 0.0003
Trainable parameters: 7,690

Epoch 001 | HEAD         | Train  15.50% ( 31/200) | Val  40.00% ( 4/10) | TLoss 2.2582 | VLoss 2.1394 | LR 3.00e-04 |   17.5s ★ BEST
Epoch 002 | HEAD         | Train  33.00% ( 66/200) | Val  40.00% ( 4/10) | TLoss 2.0962 | VLoss 1.9991 | LR 2.97e-04 |   17.7s ★ BEST
Epoch 003 | HEAD         | Train  54.50% (109/200) | Val  70.00% ( 7/10) | TLoss 1.9470 | VLoss 1.8810 | LR 2.88e-04 |   16.8s ★ BEST
Epoch 004 | HEAD         | Train  67.50% (135/200) | Val  70.00% ( 7/10) | TLoss 1.8245 | VLoss 1.7841 | LR 2.73e-04 |   17.3s ★ BEST
Epoch 005 | HEAD         | Train  75.00% (150/200) | Val  50.00% ( 5/10) | TLoss 1.6533 | VLoss 1.7142 | LR 2.53e-04 |   15.4s 
Epoch 006 | HEAD         | Train  62.50% (125/200) | Val  70.00% ( 7/10) | TLoss 1.6842 | VLoss 1.6272 | LR 2.29e-04 |   14.6s ★ BEST
Epoch 007 | HEAD         | Train  83.50% (167/200) | Val  70.0